# Subject: test_pyrepl
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Lib/test/test_pyrepl

### File: __main__.py

In [ ]:
import unittest
from test.test_pyrepl import load_tests

unittest.main()

### File: eio_test_script.py

In [ ]:
import errno
import fcntl
import os
import pty
import signal
import sys
import termios


def handler(sig, f):
    pass


def create_eio_condition():
    # SIGINT handler used to produce an EIO.
    # See https://github.com/python/cpython/issues/135329.
    try:
        master_fd, slave_fd = pty.openpty()
        child_pid = os.fork()
        if child_pid == 0:
            try:
                os.setsid()
                fcntl.ioctl(slave_fd, termios.TIOCSCTTY, 0)
                child_process_group_id = os.getpgrp()
                grandchild_pid = os.fork()
                if grandchild_pid == 0:
                    os.setpgid(0, 0)      # set process group for grandchild
                    os.dup2(slave_fd, 0)  # redirect stdin
                    if slave_fd > 2:
                        os.close(slave_fd)
                    # Fork grandchild for terminal control manipulation
                    if os.fork() == 0:
                        sys.exit(0)  # exit the child process that was just obtained
                    else:
                        try:
                            os.tcsetpgrp(0, child_process_group_id)
                        except OSError:
                            pass
                        sys.exit(0)
                else:
                    # Back to child
                    try:
                        os.setpgid(grandchild_pid, grandchild_pid)
                    except ProcessLookupError:
                        pass
                    os.tcsetpgrp(slave_fd, grandchild_pid)
                    if slave_fd > 2:
                        os.close(slave_fd)
                    os.waitpid(grandchild_pid, 0)
                    # Manipulate terminal control to create EIO condition
                    os.tcsetpgrp(master_fd, child_process_group_id)
                    # Now try to read from master - this might cause EIO
                    try:
                        os.read(master_fd, 1)
                    except OSError as e:
                        if e.errno == errno.EIO:
                            print(f"Setup created EIO condition: {e}", file=sys.stderr)
                    sys.exit(0)
            except Exception as setup_e:
                print(f"Setup error: {setup_e}", file=sys.stderr)
                sys.exit(1)
        else:
            # Parent process
            os.close(slave_fd)
            os.waitpid(child_pid, 0)
            # Now replace stdin with master_fd and try to read
            os.dup2(master_fd, 0)
            os.close(master_fd)
            # This should now trigger EIO
            print(f"Unexpectedly got input: {input()!r}", file=sys.stderr)
            sys.exit(0)
    except OSError as e:
        if e.errno == errno.EIO:
            print(f"Got EIO: {e}", file=sys.stderr)
            sys.exit(1)
        elif e.errno == errno.ENXIO:
            print(f"Got ENXIO (no such device): {e}", file=sys.stderr)
            sys.exit(1)  # Treat ENXIO as success too
        else:
            print(f"Got other OSError: errno={e.errno} {e}", file=sys.stderr)
            sys.exit(2)
    except EOFError as e:
        print(f"Got EOFError: {e}", file=sys.stderr)
        sys.exit(3)
    except Exception as e:
        print(f"Got unexpected error: {type(e).__name__}: {e}", file=sys.stderr)
        sys.exit(4)


if __name__ == "__main__":
    # Set up signal handler for coordination
    signal.signal(signal.SIGUSR1, lambda *a: create_eio_condition())
    print("READY", flush=True)
    signal.pause()

### File: support.py

In [ ]:
from code import InteractiveConsole
from functools import partial
from typing import Iterable
from unittest.mock import MagicMock

from _pyrepl.console import Console, Event
from _pyrepl.readline import ReadlineAlikeReader, ReadlineConfig
from _pyrepl.simple_interact import _strip_final_indent
from _pyrepl.utils import unbracket, ANSI_ESCAPE_SEQUENCE


class ScreenEqualMixin:
    def assert_screen_equal(
        self, reader: ReadlineAlikeReader, expected: str, clean: bool = False
    ):
        actual = clean_screen(reader) if clean else reader.screen
        expected = expected.split("\n")
        self.assertListEqual(actual, expected)


def multiline_input(reader: ReadlineAlikeReader, namespace: dict | None = None):
    saved = reader.more_lines
    try:
        reader.more_lines = partial(more_lines, namespace=namespace)
        reader.ps1 = reader.ps2 = ">>> "
        reader.ps3 = reader.ps4 = "... "
        return reader.readline()
    finally:
        reader.more_lines = saved
        reader.paste_mode = False


def more_lines(text: str, namespace: dict | None = None):
    if namespace is None:
        namespace = {}
    src = _strip_final_indent(text)
    console = InteractiveConsole(namespace, filename="<stdin>")
    try:
        code = console.compile(src, "<stdin>", "single")
    except (OverflowError, SyntaxError, ValueError):
        return False
    else:
        return code is None


def code_to_events(code: str):
    for c in code:
        yield Event(evt="key", data=c, raw=bytearray(c.encode("utf-8")))


def clean_screen(reader: ReadlineAlikeReader) -> list[str]:
    """Cleans color and console characters out of a screen output.

    This is useful for screen testing, it increases the test readability since
    it strips out all the unreadable side of the screen.
    """
    output = []
    for line in reader.screen:
        line = unbracket(line, including_content=True)
        line = ANSI_ESCAPE_SEQUENCE.sub("", line)
        for prefix in (reader.ps1, reader.ps2, reader.ps3, reader.ps4):
            if line.startswith(prefix):
                line = line[len(prefix):]
                break
        output.append(line)
    return output


def prepare_reader(console: Console, **kwargs):
    config = ReadlineConfig(readline_completer=kwargs.pop("readline_completer", None))
    reader = ReadlineAlikeReader(console=console, config=config)
    reader.more_lines = partial(more_lines, namespace=None)
    reader.paste_mode = True  # Avoid extra indents

    def get_prompt(lineno, cursor_on_line) -> str:
        return ""

    reader.get_prompt = get_prompt  # Remove prompt for easier calculations of (x, y)

    for key, val in kwargs.items():
        setattr(reader, key, val)

    return reader


def prepare_console(events: Iterable[Event], **kwargs) -> MagicMock | Console:
    console = MagicMock()
    console.get_event.side_effect = events
    console.height = 100
    console.width = 80
    for key, val in kwargs.items():
        setattr(console, key, val)
    return console


def handle_all_events(
    events, prepare_console=prepare_console, prepare_reader=prepare_reader
):
    console = prepare_console(events)
    reader = prepare_reader(console)
    try:
        while True:
            reader.handle1()
    except StopIteration:
        pass
    except KeyboardInterrupt:
        pass
    return reader, console


handle_events_narrow_console = partial(
    handle_all_events,
    prepare_console=partial(prepare_console, width=10),
)


class FakeConsole(Console):
    def __init__(self, events, encoding="utf-8") -> None:
        self.events = iter(events)
        self.encoding = encoding
        self.screen = []
        self.height = 100
        self.width = 80

    def get_event(self, block: bool = True) -> Event | None:
        return next(self.events)

    def getpending(self) -> Event:
        return self.get_event(block=False)

    def getheightwidth(self) -> tuple[int, int]:
        return self.height, self.width

    def refresh(self, screen: list[str], xy: tuple[int, int]) -> None:
        pass

    def prepare(self) -> None:
        pass

    def restore(self) -> None:
        pass

    def move_cursor(self, x: int, y: int) -> None:
        pass

    def set_cursor_vis(self, visible: bool) -> None:
        pass

    def push_char(self, char: int | bytes) -> None:
        pass

    def beep(self) -> None:
        pass

    def clear(self) -> None:
        pass

    def finish(self) -> None:
        pass

    def flushoutput(self) -> None:
        pass

    def forgetinput(self) -> None:
        pass

    def wait(self, timeout: float | None = None) -> bool:
        return True

    def repaint(self) -> None:
        pass

### File: test_eventqueue.py

In [ ]:
import tempfile
import unittest
from unittest.mock import patch
from test import support

from _pyrepl import terminfo

try:
    from _pyrepl.console import Event
    from _pyrepl import base_eventqueue
except ImportError:
    pass

try:
    from _pyrepl import unix_eventqueue
except ImportError:
    pass

try:
    from _pyrepl import windows_eventqueue
except ImportError:
    pass

class EventQueueTestBase:
    """OS-independent mixin"""
    def make_eventqueue(self) -> base_eventqueue.BaseEventQueue:
        raise NotImplementedError()

    def test_get(self):
        eq = self.make_eventqueue()
        event = Event("key", "a", b"a")
        eq.insert(event)
        self.assertEqual(eq.get(), event)

    def test_empty(self):
        eq = self.make_eventqueue()
        self.assertTrue(eq.empty())
        eq.insert(Event("key", "a", b"a"))
        self.assertFalse(eq.empty())

    def test_flush_buf(self):
        eq = self.make_eventqueue()
        eq.buf.extend(b"test")
        self.assertEqual(eq.flush_buf(), b"test")
        self.assertEqual(eq.buf, bytearray())

    def test_insert(self):
        eq = self.make_eventqueue()
        event = Event("key", "a", b"a")
        eq.insert(event)
        self.assertEqual(eq.events[0], event)

    @patch("_pyrepl.base_eventqueue.keymap")
    def test_push_with_key_in_keymap(self, mock_keymap):
        mock_keymap.compile_keymap.return_value = {"a": "b"}
        eq = self.make_eventqueue()
        eq.keymap = {b"a": "b"}
        eq.push(b"a")
        mock_keymap.compile_keymap.assert_called()
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "b")

    @patch("_pyrepl.base_eventqueue.keymap")
    def test_push_without_key_in_keymap(self, mock_keymap):
        mock_keymap.compile_keymap.return_value = {"a": "b"}
        eq = self.make_eventqueue()
        eq.keymap = {b"c": "d"}
        eq.push(b"a")
        mock_keymap.compile_keymap.assert_called()
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "a")

    @patch("_pyrepl.base_eventqueue.keymap")
    def test_push_with_keymap_in_keymap(self, mock_keymap):
        mock_keymap.compile_keymap.return_value = {"a": "b"}
        eq = self.make_eventqueue()
        eq.keymap = {b"a": {b"b": "c"}}
        eq.push(b"a")
        mock_keymap.compile_keymap.assert_called()
        self.assertTrue(eq.empty())
        eq.push(b"b")
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "c")
        eq.push(b"d")
        self.assertEqual(eq.events[1].evt, "key")
        self.assertEqual(eq.events[1].data, "d")

    @patch("_pyrepl.base_eventqueue.keymap")
    def test_push_with_keymap_in_keymap_and_escape(self, mock_keymap):
        mock_keymap.compile_keymap.return_value = {"a": "b"}
        eq = self.make_eventqueue()
        eq.keymap = {b"a": {b"b": "c"}}
        eq.push(b"a")
        mock_keymap.compile_keymap.assert_called()
        self.assertTrue(eq.empty())
        eq.flush_buf()
        eq.push(b"\033")
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "\033")
        eq.push(b"b")
        self.assertEqual(eq.events[1].evt, "key")
        self.assertEqual(eq.events[1].data, "b")

    def test_push_special_key(self):
        eq = self.make_eventqueue()
        eq.keymap = {}
        eq.push(b"\x1b")
        eq.push(b"[")
        eq.push(b"A")
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "\x1b")

    def test_push_unrecognized_escape_sequence(self):
        eq = self.make_eventqueue()
        eq.keymap = {}
        eq.push(b"\x1b")
        eq.push(b"[")
        eq.push(b"Z")
        self.assertEqual(len(eq.events), 3)
        self.assertEqual(eq.events[0].evt, "key")
        self.assertEqual(eq.events[0].data, "\x1b")
        self.assertEqual(eq.events[1].evt, "key")
        self.assertEqual(eq.events[1].data, "[")
        self.assertEqual(eq.events[2].evt, "key")
        self.assertEqual(eq.events[2].data, "Z")

    def test_push_unicode_character_as_str(self):
        eq = self.make_eventqueue()
        eq.keymap = {}
        with self.assertRaises(AssertionError):
            eq.push("ч")
        with self.assertRaises(AssertionError):
            eq.push("ñ")

    def test_push_unicode_character_two_bytes(self):
        eq = self.make_eventqueue()
        eq.keymap = {}

        encoded = "ч".encode(eq.encoding, "replace")
        self.assertEqual(len(encoded), 2)

        eq.push(encoded[0])
        e = eq.get()
        self.assertIsNone(e)

        eq.push(encoded[1])
        e = eq.get()
        self.assertEqual(e.evt, "key")
        self.assertEqual(e.data, "ч")

    def test_push_single_chars_and_unicode_character_as_str(self):
        eq = self.make_eventqueue()
        eq.keymap = {}

        def _event(evt, data, raw=None):
            r = raw if raw is not None else data.encode(eq.encoding)
            e = Event(evt, data, r)
            return e

        def _push(keys):
            for k in keys:
                eq.push(k)

        self.assertIsInstance("ñ", str)

        # If an exception happens during push, the existing events must be
        # preserved and we can continue to push.
        _push(b"b")
        with self.assertRaises(AssertionError):
            _push("ñ")
        _push(b"a")

        self.assertEqual(eq.get(), _event("key", "b"))
        self.assertEqual(eq.get(), _event("key", "a"))


class EmptyTermInfo(terminfo.TermInfo):
    def get(self, cap: str) -> bytes:
        return b""


@unittest.skipIf(support.MS_WINDOWS, "No Unix event queue on Windows")
class TestUnixEventQueue(EventQueueTestBase, unittest.TestCase):
    def setUp(self):
        self.file = tempfile.TemporaryFile()

    def tearDown(self) -> None:
        self.file.close()

    def make_eventqueue(self) -> base_eventqueue.BaseEventQueue:
        ti = EmptyTermInfo("ansi")
        return unix_eventqueue.EventQueue(self.file.fileno(), "utf-8", ti)


@unittest.skipUnless(support.MS_WINDOWS, "No Windows event queue on Unix")
class TestWindowsEventQueue(EventQueueTestBase, unittest.TestCase):
    def make_eventqueue(self) -> base_eventqueue.BaseEventQueue:
        return windows_eventqueue.EventQueue("utf-8")

### File: test_input.py

In [ ]:
import unittest

from _pyrepl.console import Event
from _pyrepl.input import KeymapTranslator


class KeymapTranslatorTests(unittest.TestCase):
    def test_push_single_key(self):
        keymap = [("a", "command_a")]
        translator = KeymapTranslator(keymap)
        evt = Event("key", "a")
        translator.push(evt)
        result = translator.get()
        self.assertEqual(result, ("command_a", ["a"]))

    def test_push_multiple_keys(self):
        keymap = [("ab", "command_ab")]
        translator = KeymapTranslator(keymap)
        evt1 = Event("key", "a")
        evt2 = Event("key", "b")
        translator.push(evt1)
        translator.push(evt2)
        result = translator.get()
        self.assertEqual(result, ("command_ab", ["a", "b"]))

    def test_push_invalid_key(self):
        keymap = [("a", "command_a")]
        translator = KeymapTranslator(keymap)
        evt = Event("key", "b")
        translator.push(evt)
        result = translator.get()
        self.assertEqual(result, (None, ["b"]))

    def test_push_invalid_key_with_stack(self):
        keymap = [("ab", "command_ab")]
        translator = KeymapTranslator(keymap)
        evt1 = Event("key", "a")
        evt2 = Event("key", "c")
        translator.push(evt1)
        translator.push(evt2)
        result = translator.get()
        self.assertEqual(result, (None, ["a", "c"]))

    def test_push_character_key(self):
        keymap = [("a", "command_a")]
        translator = KeymapTranslator(keymap)
        evt = Event("key", "a")
        translator.push(evt)
        result = translator.get()
        self.assertEqual(result, ("command_a", ["a"]))

    def test_push_character_key_with_stack(self):
        keymap = [("ab", "command_ab")]
        translator = KeymapTranslator(keymap)
        evt1 = Event("key", "a")
        evt2 = Event("key", "b")
        evt3 = Event("key", "c")
        translator.push(evt1)
        translator.push(evt2)
        translator.push(evt3)
        result = translator.get()
        self.assertEqual(result, ("command_ab", ["a", "b"]))

    def test_push_transition_key(self):
        keymap = [("a", {"b": "command_ab"})]
        translator = KeymapTranslator(keymap)
        evt1 = Event("key", "a")
        evt2 = Event("key", "b")
        translator.push(evt1)
        translator.push(evt2)
        result = translator.get()
        self.assertEqual(result, ("command_ab", ["a", "b"]))

    def test_push_transition_key_interrupted(self):
        keymap = [("a", {"b": "command_ab"})]
        translator = KeymapTranslator(keymap)
        evt1 = Event("key", "a")
        evt2 = Event("key", "c")
        evt3 = Event("key", "b")
        translator.push(evt1)
        translator.push(evt2)
        translator.push(evt3)
        result = translator.get()
        self.assertEqual(result, (None, ["a", "c"]))

    def test_push_invalid_key_with_unicode_category(self):
        keymap = [("a", "command_a")]
        translator = KeymapTranslator(keymap)
        evt = Event("key", "\u0003")  # Control character
        translator.push(evt)
        result = translator.get()
        self.assertEqual(result, (None, ["\u0003"]))

    def test_empty(self):
        keymap = [("a", "command_a")]
        translator = KeymapTranslator(keymap)
        self.assertTrue(translator.empty())
        evt = Event("key", "a")
        translator.push(evt)
        self.assertFalse(translator.empty())
        translator.get()
        self.assertTrue(translator.empty())

### File: test_interact.py

In [ ]:
import contextlib
import io
import unittest
from unittest.mock import patch
from textwrap import dedent

from test.support import force_not_colorized

from _pyrepl.console import InteractiveColoredConsole
from _pyrepl.simple_interact import _more_lines

class TestSimpleInteract(unittest.TestCase):
    def test_multiple_statements(self):
        namespace = {}
        code = dedent("""\
        class A:
            def foo(self):


                pass

        class B:
            def bar(self):
                pass

        a = 1
        a
        """)
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        f = io.StringIO()
        with (
            patch.object(InteractiveColoredConsole, "showsyntaxerror") as showsyntaxerror,
            patch.object(InteractiveColoredConsole, "runsource", wraps=console.runsource) as runsource,
            contextlib.redirect_stdout(f),
        ):
            more = console.push(code, filename="<stdin>", _symbol="single")  # type: ignore[call-arg]
        self.assertFalse(more)
        showsyntaxerror.assert_not_called()


    def test_multiple_statements_output(self):
        namespace = {}
        code = dedent("""\
        b = 1
        b
        a = 1
        a
        """)
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            more = console.push(code, filename="<stdin>", _symbol="single")  # type: ignore[call-arg]
        self.assertFalse(more)
        self.assertEqual(f.getvalue(), "1\n")

    @force_not_colorized
    def test_multiple_statements_fail_early(self):
        console = InteractiveColoredConsole()
        code = dedent("""\
        raise Exception('foobar')
        print('spam', 'eggs', sep='&')
        """)
        f = io.StringIO()
        with contextlib.redirect_stderr(f):
            console.runsource(code)
        self.assertIn('Exception: foobar', f.getvalue())
        self.assertNotIn('spam&eggs', f.getvalue())

    def test_empty(self):
        namespace = {}
        code = ""
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            more = console.push(code, filename="<stdin>", _symbol="single")  # type: ignore[call-arg]
        self.assertFalse(more)
        self.assertEqual(f.getvalue(), "")

    def test_runsource_compiles_and_runs_code(self):
        console = InteractiveColoredConsole()
        source = "print('Hello, world!')"
        with patch.object(console, "runcode") as mock_runcode:
            console.runsource(source)
            mock_runcode.assert_called_once()

    def test_runsource_returns_false_for_successful_compilation(self):
        console = InteractiveColoredConsole()
        source = "print('Hello, world!')"
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            result = console.runsource(source)
        self.assertFalse(result)

    @force_not_colorized
    def test_runsource_returns_false_for_failed_compilation(self):
        console = InteractiveColoredConsole()
        source = "print('Hello, world!'"
        f = io.StringIO()
        with contextlib.redirect_stderr(f):
            result = console.runsource(source)
        self.assertFalse(result)
        self.assertIn('SyntaxError', f.getvalue())

    @force_not_colorized
    def test_runsource_show_syntax_error_location(self):
        console = InteractiveColoredConsole()
        source = "def f(x, x): ..."
        f = io.StringIO()
        with contextlib.redirect_stderr(f):
            result = console.runsource(source)
        self.assertFalse(result)
        r = """
    def f(x, x): ...
             ^
SyntaxError: duplicate parameter 'x' in function definition"""
        self.assertIn(r, f.getvalue())

    def test_runsource_shows_syntax_error_for_failed_compilation(self):
        console = InteractiveColoredConsole()
        source = "print('Hello, world!'"
        with patch.object(console, "showsyntaxerror") as mock_showsyntaxerror:
            console.runsource(source)
            mock_showsyntaxerror.assert_called_once()
        source = dedent("""\
        match 1:
            case {0: _, 0j: _}:
                pass
        """)
        with patch.object(console, "showsyntaxerror") as mock_showsyntaxerror:
            console.runsource(source)
            mock_showsyntaxerror.assert_called_once()

    def test_runsource_survives_null_bytes(self):
        console = InteractiveColoredConsole()
        source = "\x00\n"
        f = io.StringIO()
        with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
            result = console.runsource(source)
        self.assertFalse(result)
        self.assertIn("source code string cannot contain null bytes", f.getvalue())

    def test_no_active_future(self):
        console = InteractiveColoredConsole()
        source = dedent("""\
        x: int = 1
        print(__annotate__(1))
        """)
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            result = console.runsource(source)
        self.assertFalse(result)
        self.assertEqual(f.getvalue(), "{'x': <class 'int'>}\n")

    def test_future_annotations(self):
        console = InteractiveColoredConsole()
        source = dedent("""\
        from __future__ import annotations
        def g(x: int): ...
        print(g.__annotations__)
        """)
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            result = console.runsource(source)
        self.assertFalse(result)
        self.assertEqual(f.getvalue(), "{'x': 'int'}\n")

    def test_future_barry_as_flufl(self):
        console = InteractiveColoredConsole()
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            result = console.runsource("from __future__ import barry_as_FLUFL\n")
            result = console.runsource("""print("black" <> 'blue')\n""")
        self.assertFalse(result)
        self.assertEqual(f.getvalue(), "True\n")


class TestMoreLines(unittest.TestCase):
    def test_invalid_syntax_single_line(self):
        namespace = {}
        code = "if foo"
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_empty_line(self):
        namespace = {}
        code = ""
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_valid_single_statement(self):
        namespace = {}
        code = "foo = 1"
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_multiline_single_assignment(self):
        namespace = {}
        code = dedent("""\
        foo = [
            1,
            2,
            3,
        ]""")
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_multiline_single_block(self):
        namespace = {}
        code = dedent("""\
        def foo():
            '''docs'''

            return 1""")
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertTrue(_more_lines(console, code))

    def test_multiple_statements_single_line(self):
        namespace = {}
        code = "foo = 1;bar = 2"
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_multiple_statements(self):
        namespace = {}
        code = dedent("""\
        import time

        foo = 1""")
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertTrue(_more_lines(console, code))

    def test_multiple_blocks(self):
        namespace = {}
        code = dedent("""\
        from dataclasses import dataclass

        @dataclass
        class Point:
            x: float
            y: float""")
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertTrue(_more_lines(console, code))

    def test_multiple_blocks_empty_newline(self):
        namespace = {}
        code = dedent("""\
        from dataclasses import dataclass

        @dataclass
        class Point:
            x: float
            y: float
        """)
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_multiple_blocks_indented_newline(self):
        namespace = {}
        code = (
            "from dataclasses import dataclass\n"
            "\n"
            "@dataclass\n"
            "class Point:\n"
            "    x: float\n"
            "    y: float\n"
            "    "
        )
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertFalse(_more_lines(console, code))

    def test_incomplete_statement(self):
        namespace = {}
        code = "if foo:"
        console = InteractiveColoredConsole(namespace, filename="<stdin>")
        self.assertTrue(_more_lines(console, code))

### File: test_keymap.py

In [ ]:
import string
import unittest

from _pyrepl.keymap import _keynames, _escapes, parse_keys, compile_keymap, KeySpecError


class TestParseKeys(unittest.TestCase):
    def test_single_character(self):
        """Ensure that single ascii characters or single digits are parsed as single characters."""
        test_cases = [(key, [key]) for key in string.ascii_letters + string.digits]
        for test_key, expected_keys in test_cases:
            with self.subTest(f"{test_key} should be parsed as {expected_keys}"):
                self.assertEqual(parse_keys(test_key), expected_keys)

    def test_keynames(self):
        """Ensure that keynames are parsed to their corresponding mapping.

        A keyname is expected to be of the following form: \\<keyname> such as \\<left>
        which would get parsed as "left".
        """
        test_cases = [(f"\\<{keyname}>", [parsed_keyname]) for keyname, parsed_keyname in _keynames.items()]
        for test_key, expected_keys in test_cases:
            with self.subTest(f"{test_key} should be parsed as {expected_keys}"):
                self.assertEqual(parse_keys(test_key), expected_keys)

    def test_escape_sequences(self):
        """Ensure that escaping sequences are parsed to their corresponding mapping."""
        test_cases = [(f"\\{escape}", [parsed_escape]) for escape, parsed_escape in _escapes.items()]
        for test_key, expected_keys in test_cases:
            with self.subTest(f"{test_key} should be parsed as {expected_keys}"):
                self.assertEqual(parse_keys(test_key), expected_keys)

    def test_control_sequences(self):
        """Ensure that supported control sequences are parsed successfully."""
        keys = ["@", "[", "]", "\\", "^", "_", "\\<space>", "\\<delete>"]
        keys.extend(string.ascii_letters)
        test_cases = [(f"\\C-{key}", chr(ord(key) & 0x1F)) for key in []]
        for test_key, expected_keys in test_cases:
            with self.subTest(f"{test_key} should be parsed as {expected_keys}"):
                self.assertEqual(parse_keys(test_key), expected_keys)

    def test_meta_sequences(self):
        self.assertEqual(parse_keys("\\M-a"), ["\033", "a"])
        self.assertEqual(parse_keys("\\M-b"), ["\033", "b"])
        self.assertEqual(parse_keys("\\M-c"), ["\033", "c"])

    def test_combinations(self):
        self.assertEqual(parse_keys("\\C-a\\n\\<up>"), ["\x01", "\n", "up"])
        self.assertEqual(parse_keys("\\M-a\\t\\<down>"), ["\033", "a", "\t", "down"])

    def test_keyspec_errors(self):
        cases = [
            ("\\Ca", "\\C must be followed by `-'"),
            ("\\ca", "\\C must be followed by `-'"),
            ("\\C-\\C-", "doubled \\C-"),
            ("\\Ma", "\\M must be followed by `-'"),
            ("\\ma", "\\M must be followed by `-'"),
            ("\\M-\\M-", "doubled \\M-"),
            ("\\<left", "unterminated \\<"),
            ("\\<unsupported>", "unrecognised keyname"),
            ("\\大", "unknown backslash escape"),
            ("\\C-\\<backspace>", "\\C- followed by invalid key")
        ]
        for test_keys, expected_err in cases:
            with self.subTest(f"{test_keys} should give error {expected_err}"):
                with self.assertRaises(KeySpecError) as e:
                    parse_keys(test_keys)
                self.assertIn(expected_err, str(e.exception))

    def test_index_errors(self):
        test_cases = ["\\", "\\C", "\\C-\\C"]
        for test_keys in test_cases:
            with self.assertRaises(IndexError):
                parse_keys(test_keys)


class TestCompileKeymap(unittest.TestCase):
    def test_empty_keymap(self):
        keymap = {}
        result = compile_keymap(keymap)
        self.assertEqual(result, {})

    def test_single_keymap(self):
        keymap = {b"a": "action"}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": "action"})

    def test_nested_keymap(self):
        keymap = {b"a": {b"b": "action"}}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": {b"b": "action"}})

    def test_empty_value(self):
        keymap = {b"a": {b"": "action"}}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": {b"": "action"}})

    def test_multiple_empty_values(self):
        keymap = {b"a": {b"": "action1", b"b": "action2"}}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": {b"": "action1", b"b": "action2"}})

    def test_multiple_keymaps(self):
        keymap = {b"a": {b"b": "action1", b"c": "action2"}}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": {b"b": "action1", b"c": "action2"}})

    def test_nested_multiple_keymaps(self):
        keymap = {b"a": {b"b": {b"c": "action"}}}
        result = compile_keymap(keymap)
        self.assertEqual(result, {b"a": {b"b": {b"c": "action"}}})

    def test_clashing_definitions(self):
        km = {b'a': 'c', b'a' + b'b': 'd'}
        with self.assertRaises(KeySpecError):
            compile_keymap(km)

    def test_non_bytes_key(self):
        with self.assertRaises(TypeError):
            compile_keymap({123: 'a'})

### File: test_pyrepl.py

In [ ]:
import importlib
import io
import itertools
import os
import pathlib
import re
import rlcompleter
import select
import subprocess
import sys
import tempfile
from pkgutil import ModuleInfo
from unittest import TestCase, skipUnless, skipIf, SkipTest
from unittest.mock import patch
from test.support import force_not_colorized, make_clean_env, Py_DEBUG
from test.support import has_subprocess_support, SHORT_TIMEOUT, STDLIB_DIR
from test.support.import_helper import import_module
from test.support.os_helper import EnvironmentVarGuard, unlink

from .support import (
    FakeConsole,
    ScreenEqualMixin,
    handle_all_events,
    handle_events_narrow_console,
    more_lines,
    multiline_input,
    code_to_events,
)
from _pyrepl.console import Event
from _pyrepl._module_completer import (
    ImportParser,
    ModuleCompleter,
    HARDCODED_SUBMODULES,
)
from _pyrepl.readline import (
    ReadlineAlikeReader,
    ReadlineConfig,
    _ReadlineWrapper,
)
from _pyrepl.readline import multiline_input as readline_multiline_input

try:
    import pty
except ImportError:
    pty = None


class ReplTestCase(TestCase):
    def setUp(self):
        if not has_subprocess_support:
            raise SkipTest("test module requires subprocess")

    def run_repl(
        self,
        repl_input: str | list[str],
        env: dict | None = None,
        *,
        cmdline_args: list[str] | None = None,
        cwd: str | None = None,
        skip: bool = False,
        timeout: float = SHORT_TIMEOUT,
        exit_on_output: str | None = None,
    ) -> tuple[str, int]:
        temp_dir = None
        if cwd is None:
            temp_dir = tempfile.TemporaryDirectory(ignore_cleanup_errors=True)
            cwd = temp_dir.name
        try:
            return self._run_repl(
                repl_input,
                env=env,
                cmdline_args=cmdline_args,
                cwd=cwd,
                skip=skip,
                timeout=timeout,
                exit_on_output=exit_on_output,
            )
        finally:
            if temp_dir is not None:
                temp_dir.cleanup()

    def _run_repl(
        self,
        repl_input: str | list[str],
        *,
        env: dict | None,
        cmdline_args: list[str] | None,
        cwd: str,
        skip: bool,
        timeout: float,
        exit_on_output: str | None,
    ) -> tuple[str, int]:
        assert pty
        master_fd, slave_fd = pty.openpty()
        cmd = [sys.executable, "-i", "-u"]
        if env is None:
            cmd.append("-I")
        elif "PYTHON_HISTORY" not in env:
            env["PYTHON_HISTORY"] = os.path.join(cwd, ".regrtest_history")
        if cmdline_args is not None:
            cmd.extend(cmdline_args)

        try:
            import termios
        except ModuleNotFoundError:
            pass
        else:
            term_attr = termios.tcgetattr(slave_fd)
            term_attr[6][termios.VREPRINT] = 0  # pass through CTRL-R
            term_attr[6][termios.VINTR] = 0  # pass through CTRL-C
            termios.tcsetattr(slave_fd, termios.TCSANOW, term_attr)

        process = subprocess.Popen(
            cmd,
            stdin=slave_fd,
            stdout=slave_fd,
            stderr=slave_fd,
            cwd=cwd,
            text=True,
            close_fds=True,
            env=env if env else os.environ,
        )
        os.close(slave_fd)
        if isinstance(repl_input, list):
            repl_input = "\n".join(repl_input) + "\n"
        os.write(master_fd, repl_input.encode("utf-8"))

        output = []
        while select.select([master_fd], [], [], timeout)[0]:
            try:
                data = os.read(master_fd, 1024).decode("utf-8")
                if not data:
                    break
            except OSError:
                break
            output.append(data)
            if exit_on_output is not None:
                output = ["".join(output)]
                if exit_on_output in output[0]:
                    process.kill()
                    break
        else:
            os.close(master_fd)
            process.kill()
            process.wait(timeout=timeout)
            self.fail(f"Timeout while waiting for output, got: {''.join(output)}")

        os.close(master_fd)
        try:
            exit_code = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            process.kill()
            exit_code = process.wait()
        output = "".join(output)
        if skip and "can't use pyrepl" in output:
            self.skipTest("pyrepl not available")
        return output, exit_code


class TestCursorPosition(TestCase):
    def prepare_reader(self, events):
        console = FakeConsole(events)
        config = ReadlineConfig(readline_completer=None)
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    def test_up_arrow_simple(self):
        # fmt: off
        code = (
            "def f():\n"
            "  ...\n"
        )
        # fmt: on
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
            ],
        )

        reader, console = handle_all_events(events)
        self.assertEqual(reader.cxy, (0, 1))
        console.move_cursor.assert_called_once_with(0, 1)

    def test_down_arrow_end_of_input(self):
        # fmt: off
        code = (
            "def f():\n"
            "  ...\n"
        )
        # fmt: on
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )

        reader, console = handle_all_events(events)
        self.assertEqual(reader.cxy, (0, 2))
        console.move_cursor.assert_called_once_with(0, 2)

    def test_left_arrow_simple(self):
        events = itertools.chain(
            code_to_events("11+11"),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
            ],
        )

        reader, console = handle_all_events(events)
        self.assertEqual(reader.cxy, (4, 0))
        console.move_cursor.assert_called_once_with(4, 0)

    def test_right_arrow_end_of_line(self):
        events = itertools.chain(
            code_to_events("11+11"),
            [
                Event(evt="key", data="right", raw=bytearray(b"\x1bOC")),
            ],
        )

        reader, console = handle_all_events(events)
        self.assertEqual(reader.cxy, (5, 0))
        console.move_cursor.assert_called_once_with(5, 0)

    def test_cursor_position_simple_character(self):
        events = itertools.chain(code_to_events("k"))

        reader, _ = handle_all_events(events)
        self.assertEqual(reader.pos, 1)

        # 1 for simple character
        self.assertEqual(reader.cxy, (1, 0))

    def test_cursor_position_double_width_character(self):
        events = itertools.chain(code_to_events("樂"))

        reader, _ = handle_all_events(events)
        self.assertEqual(reader.pos, 1)

        # 2 for wide character
        self.assertEqual(reader.cxy, (2, 0))

    def test_cursor_position_double_width_character_move_left(self):
        events = itertools.chain(
            code_to_events("樂"),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
            ],
        )

        reader, _ = handle_all_events(events)
        self.assertEqual(reader.pos, 0)
        self.assertEqual(reader.cxy, (0, 0))

    def test_cursor_position_double_width_character_move_left_right(self):
        events = itertools.chain(
            code_to_events("樂"),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="right", raw=bytearray(b"\x1bOC")),
            ],
        )

        reader, _ = handle_all_events(events)
        self.assertEqual(reader.pos, 1)

        # 2 for wide character
        self.assertEqual(reader.cxy, (2, 0))

    def test_cursor_position_double_width_characters_move_up(self):
        for_loop = "for _ in _:"

        # fmt: off
        code = (
           f"{for_loop}\n"
            "  ' 可口可乐; 可口可樂'"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
            ],
        )

        reader, _ = handle_all_events(events)

        # cursor at end of first line
        self.assertEqual(reader.pos, len(for_loop))
        self.assertEqual(reader.cxy, (len(for_loop), 0))

    def test_cursor_position_double_width_characters_move_up_down(self):
        for_loop = "for _ in _:"

        # fmt: off
        code = (
           f"{for_loop}\n"
            "  ' 可口可乐; 可口可樂'"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )

        reader, _ = handle_all_events(events)

        # cursor here (showing 2nd line only):
        # <  ' 可口可乐; 可口可樂'>
        #              ^
        self.assertEqual(reader.pos, 19)
        self.assertEqual(reader.cxy, (10, 1))

    def test_cursor_position_multiple_double_width_characters_move_left(self):
        events = itertools.chain(
            code_to_events("' 可口可乐; 可口可樂'"),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
            ],
        )

        reader, _ = handle_all_events(events)
        self.assertEqual(reader.pos, 10)

        # 1 for quote, 1 for space, 2 per wide character,
        # 1 for semicolon, 1 for space, 2 per wide character
        self.assertEqual(reader.cxy, (16, 0))

    def test_cursor_position_move_up_to_eol(self):
        first_line = "for _ in _:"
        second_line = "  hello"

        # fmt: off
        code = (
            f"{first_line}\n"
            f"{second_line}\n"
             "  h\n"
             "  hel"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
            ],
        )

        reader, _ = handle_all_events(events)

        # Cursor should be at end of line 1, even though line 2 is shorter
        # for _ in _:
        #   hello
        #   h
        #   hel
        self.assertEqual(
            reader.pos, len(first_line) + len(second_line) + 1
        )  # +1 for newline
        self.assertEqual(reader.cxy, (len(second_line), 1))

    def test_cursor_position_move_down_to_eol(self):
        last_line = "  hel"

        # fmt: off
        code = (
            "for _ in _:\n"
            "  hello\n"
            "  h\n"
           f"{last_line}"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )

        reader, _ = handle_all_events(events)

        # Cursor should be at end of line 3, even though line 2 is shorter
        # for _ in _:
        #   hello
        #   h
        #   hel
        self.assertEqual(reader.pos, len(code))
        self.assertEqual(reader.cxy, (len(last_line), 3))

    def test_cursor_position_multiple_mixed_lines_move_up(self):
        # fmt: off
        code = (
            "def foo():\n"
            "  x = '可口可乐; 可口可樂'\n"
            "  y = 'abckdfjskldfjslkdjf'"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            13 * [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],
            [Event(evt="key", data="up", raw=bytearray(b"\x1bOA"))],
        )

        reader, _ = handle_all_events(events)

        # By moving left, we're before the s:
        # y = 'abckdfjskldfjslkdjf'
        #             ^
        # And we should move before the semi-colon despite the different offset
        # x = '可口可乐; 可口可樂'
        #            ^
        self.assertEqual(reader.pos, 22)
        self.assertEqual(reader.cxy, (15, 1))

    def test_cursor_position_after_wrap_and_move_up(self):
        # fmt: off
        code = (
            "def foo():\n"
            "  hello"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
            ],
        )
        reader, _ = handle_events_narrow_console(events)

        # The code looks like this:
        # def foo()\
        # :
        #   hello
        # After moving up we should be after the colon in line 2
        self.assertEqual(reader.pos, 10)
        self.assertEqual(reader.cxy, (1, 1))


class TestPyReplAutoindent(TestCase):
    def prepare_reader(self, events):
        console = FakeConsole(events)
        config = ReadlineConfig(readline_completer=None)
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    def test_auto_indent_default(self):
        # fmt: off
        input_code = (
            "def f():\n"
                "pass\n\n"
        )

        output_code = (
            "def f():\n"
            "    pass\n"
            "    "
        )
        # fmt: on

        events = code_to_events(input_code)
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_auto_indent_continuation(self):
        # auto indenting according to previous user indentation
        # fmt: off
        events = itertools.chain(
            code_to_events("def f():\n"),
            # add backspace to delete default auto-indent
            [
                Event(evt="key", data="backspace", raw=bytearray(b"\x7f")),
            ],
            code_to_events(
                "  pass\n"
                  "pass\n\n"
            ),
        )

        output_code = (
            "def f():\n"
            "  pass\n"
            "  pass\n"
            "  "
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_auto_indent_prev_block(self):
        # auto indenting according to indentation in different block
        # fmt: off
        events = itertools.chain(
            code_to_events("def f():\n"),
            # add backspace to delete default auto-indent
            [
                Event(evt="key", data="backspace", raw=bytearray(b"\x7f")),
            ],
            code_to_events(
                "  pass\n"
                "pass\n\n"
            ),
            code_to_events(
                "def g():\n"
                  "pass\n\n"
            ),
        )

        output_code = (
            "def g():\n"
            "  pass\n"
            "  "
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output1 = multiline_input(reader)
        output2 = multiline_input(reader)
        self.assertEqual(output2, output_code)

    def test_auto_indent_multiline(self):
        # fmt: off
        events = itertools.chain(
            code_to_events(
                "def f():\n"
                    "pass"
            ),
            [
                # go to the end of the first line
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\x05", raw=bytearray(b"\x1bO5")),
                # new line should be autoindented
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
            code_to_events(
                "pass"
            ),
            [
                # go to end of last line
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="\x05", raw=bytearray(b"\x1bO5")),
                # double newline to terminate the block
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        output_code = (
            "def f():\n"
            "    pass\n"
            "    pass\n"
            "    "
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_auto_indent_with_comment(self):
        # fmt: off
        events = code_to_events(
            "def f():  # foo\n"
                "pass\n\n"
        )

        output_code = (
            "def f():  # foo\n"
            "    pass\n"
            "    "
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_auto_indent_with_multicomment(self):
        # fmt: off
        events = code_to_events(
            "def f():  ## foo\n"
                "pass\n\n"
        )

        output_code = (
            "def f():  ## foo\n"
            "    pass\n"
            "    "
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_auto_indent_ignore_comments(self):
        # fmt: off
        events = code_to_events(
            "pass  #:\n"
        )

        output_code = (
            "pass  #:"
        )
        # fmt: on

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)


class TestPyReplOutput(ScreenEqualMixin, TestCase):
    def prepare_reader(self, events):
        console = FakeConsole(events)
        config = ReadlineConfig(readline_completer=None)
        reader = ReadlineAlikeReader(console=console, config=config)
        reader.can_colorize = False
        return reader

    def test_stdin_is_tty(self):
        # Used during test log analysis to figure out if a TTY was available.
        try:
            if os.isatty(sys.stdin.fileno()):
                return
        except OSError as ose:
            self.skipTest(f"stdin tty check failed: {ose}")
        else:
            self.skipTest("stdin is not a tty")

    def test_stdout_is_tty(self):
        # Used during test log analysis to figure out if a TTY was available.
        try:
            if os.isatty(sys.stdout.fileno()):
                return
        except OSError as ose:
            self.skipTest(f"stdout tty check failed: {ose}")
        else:
            self.skipTest("stdout is not a tty")

    def test_basic(self):
        reader = self.prepare_reader(code_to_events("1+1\n"))

        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)

    def test_get_line_buffer_returns_str(self):
        reader = self.prepare_reader(code_to_events("\n"))
        wrapper = _ReadlineWrapper(f_in=None, f_out=None, reader=reader)
        self.assertIs(type(wrapper.get_line_buffer()), str)

    def test_multiline_edit(self):
        events = itertools.chain(
            code_to_events("def f():\n...\n\n"),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="backspace", raw=bytearray(b"\x08")),
                Event(evt="key", data="g", raw=bytearray(b"g")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="backspace", raw=bytearray(b"\x08")),
                Event(evt="key", data="delete", raw=bytearray(b"\x7F")),
                Event(evt="key", data="right", raw=bytearray(b"g")),
                Event(evt="key", data="backspace", raw=bytearray(b"\x08")),
                Event(evt="key", data="p", raw=bytearray(b"p")),
                Event(evt="key", data="a", raw=bytearray(b"a")),
                Event(evt="key", data="s", raw=bytearray(b"s")),
                Event(evt="key", data="s", raw=bytearray(b"s")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )
        reader = self.prepare_reader(events)

        output = multiline_input(reader)
        expected = "def f():\n    ...\n    "
        self.assertEqual(output, expected)
        self.assert_screen_equal(reader, expected, clean=True)
        output = multiline_input(reader)
        expected = "def g():\n    pass\n    "
        self.assertEqual(output, expected)
        self.assert_screen_equal(reader, expected, clean=True)

    def test_history_navigation_with_up_arrow(self):
        events = itertools.chain(
            code_to_events("1+1\n2+2\n"),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        reader = self.prepare_reader(events)

        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "2+2")
        self.assert_screen_equal(reader, "2+2", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "2+2")
        self.assert_screen_equal(reader, "2+2", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)

    def test_history_with_multiline_entries(self):
        code = "def foo():\nx = 1\ny = 2\nz = 3\n\ndef bar():\nreturn 42\n\n"
        events = list(itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ]
        ))

        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        output = multiline_input(reader)
        output = multiline_input(reader)
        expected = "def foo():\n    x = 1\n    y = 2\n    z = 3\n    "
        self.assert_screen_equal(reader, expected, clean=True)
        self.assertEqual(output, expected)


    def test_history_navigation_with_down_arrow(self):
        events = itertools.chain(
            code_to_events("1+1\n2+2\n"),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )

        reader = self.prepare_reader(events)

        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)

    def test_history_search(self):
        events = itertools.chain(
            code_to_events("1+1\n2+2\n3+3\n"),
            [
                Event(evt="key", data="\x12", raw=bytearray(b"\x12")),
                Event(evt="key", data="1", raw=bytearray(b"1")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        reader = self.prepare_reader(events)

        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "2+2")
        self.assert_screen_equal(reader, "2+2", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "3+3")
        self.assert_screen_equal(reader, "3+3", clean=True)
        output = multiline_input(reader)
        self.assertEqual(output, "1+1")
        self.assert_screen_equal(reader, "1+1", clean=True)

    def test_control_character(self):
        events = code_to_events("c\x1d\n")
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, "c\x1d")
        self.assert_screen_equal(reader, "c\x1d", clean=True)

    def test_history_search_backward(self):
        # Test <page up> history search backward with "imp" input
        events = itertools.chain(
            code_to_events("import os\n"),
            code_to_events("imp"),
            [
                Event(evt='key', data='page up', raw=bytearray(b'\x1b[5~')),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        # fill the history
        reader = self.prepare_reader(events)
        multiline_input(reader)

        # search for "imp" in history
        output = multiline_input(reader)
        self.assertEqual(output, "import os")
        self.assert_screen_equal(reader, "import os", clean=True)

    def test_history_search_backward_empty(self):
        # Test <page up> history search backward with an empty input
        events = itertools.chain(
            code_to_events("import os\n"),
            [
                Event(evt='key', data='page up', raw=bytearray(b'\x1b[5~')),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        # fill the history
        reader = self.prepare_reader(events)
        multiline_input(reader)

        # search backward in history
        output = multiline_input(reader)
        self.assertEqual(output, "import os")
        self.assert_screen_equal(reader, "import os", clean=True)


class TestPyReplCompleter(TestCase):
    def prepare_reader(self, events, namespace):
        console = FakeConsole(events)
        config = ReadlineConfig()
        config.readline_completer = rlcompleter.Completer(namespace).complete
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    @patch("rlcompleter._readline_available", False)
    def test_simple_completion(self):
        events = code_to_events("os.getpid\t\n")

        namespace = {"os": os}
        reader = self.prepare_reader(events, namespace)

        output = multiline_input(reader, namespace)
        self.assertEqual(output, "os.getpid()")

    def test_completion_with_many_options(self):
        # Test with something that initially displays many options
        # and then complete from one of them. The first time tab is
        # pressed, the options are displayed (which corresponds to
        # when the repl shows [ not unique ]) and the second completes
        # from one of them.
        events = code_to_events("os.\t\tO_AP\t\n")

        namespace = {"os": os}
        reader = self.prepare_reader(events, namespace)

        output = multiline_input(reader, namespace)
        self.assertEqual(output, "os.O_APPEND")

    def test_empty_namespace_completion(self):
        events = code_to_events("os.geten\t\n")
        namespace = {}
        reader = self.prepare_reader(events, namespace)

        output = multiline_input(reader, namespace)
        self.assertEqual(output, "os.geten")

    def test_global_namespace_completion(self):
        events = code_to_events("py\t\n")
        namespace = {"python": None}
        reader = self.prepare_reader(events, namespace)
        output = multiline_input(reader, namespace)
        self.assertEqual(output, "python")

    def test_up_down_arrow_with_completion_menu(self):
        """Up arrow in the middle of unfinished tab completion when the menu is displayed
        should work and trigger going back in history. Down arrow should subsequently
        get us back to the incomplete command."""
        code = "import os\nos.\t\t"
        namespace = {"os": os}

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
            code_to_events("\n"),
        )
        reader = self.prepare_reader(events, namespace=namespace)
        output = multiline_input(reader, namespace)
        # This is the first line, nothing to see here
        self.assertEqual(output, "import os")
        # This is the second line. We pressed up and down arrows
        # so we should end up where we were when we initiated tab completion.
        output = multiline_input(reader, namespace)
        self.assertEqual(output, "os.")

    @patch("_pyrepl.readline._ReadlineWrapper.get_reader")
    @patch("sys.stderr", new_callable=io.StringIO)
    def test_completion_with_warnings(self, mock_stderr, mock_get_reader):
        class Dummy:
            @property
            def test_func(self):
                import warnings

                warnings.warn("warnings\n")
                return None

        dummy = Dummy()
        events = code_to_events("dummy.test_func.\t\n\n")
        namespace = {"dummy": dummy}
        reader = self.prepare_reader(events, namespace)
        mock_get_reader.return_value = reader
        output = readline_multiline_input(more_lines, ">>>", "...")
        self.assertEqual(output, "dummy.test_func.__")
        self.assertEqual(mock_stderr.getvalue(), "")


class TestPyReplModuleCompleter(TestCase):
    def setUp(self):
        # Make iter_modules() search only the standard library.
        # This makes the test more reliable in case there are
        # other user packages/scripts on PYTHONPATH which can
        # interfere with the completions.
        lib_path = os.path.dirname(importlib.__path__[0])
        self._saved_sys_path = sys.path
        sys.path = [lib_path]

    def tearDown(self):
        sys.path = self._saved_sys_path

    def prepare_reader(self, events, namespace):
        console = FakeConsole(events)
        config = ReadlineConfig()
        config.module_completer = ModuleCompleter(namespace)
        config.readline_completer = rlcompleter.Completer(namespace).complete
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    def test_import_completions(self):
        cases = (
            ("import path\t\n", "import pathlib"),
            ("import importlib.\t\tres\t\n", "import importlib.resources"),
            ("import importlib.resources.\t\ta\t\n", "import importlib.resources.abc"),
            ("import foo, impo\t\n", "import foo, importlib"),
            ("import foo as bar, impo\t\n", "import foo as bar, importlib"),
            ("from impo\t\n", "from importlib"),
            ("from importlib.res\t\n", "from importlib.resources"),
            ("from importlib.\t\tres\t\n", "from importlib.resources"),
            ("from importlib.resources.ab\t\n", "from importlib.resources.abc"),
            ("from importlib import mac\t\n", "from importlib import machinery"),
            ("from importlib import res\t\n", "from importlib import resources"),
            ("from importlib.res\t import a\t\n", "from importlib.resources import abc"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    @patch("pkgutil.iter_modules", lambda: [ModuleInfo(None, "public", True),
                                            ModuleInfo(None, "_private", True)])
    @patch("sys.builtin_module_names", ())
    def test_private_completions(self):
        cases = (
            # Return public methods by default
            ("import \t\n", "import public"),
            ("from \t\n", "from public"),
            # Return private methods if explicitly specified
            ("import _\t\n", "import _private"),
            ("from _\t\n", "from _private"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    @patch(
        "_pyrepl._module_completer.ModuleCompleter.iter_submodules",
        lambda *_: [
            ModuleInfo(None, "public", True),
            ModuleInfo(None, "_private", True),
        ],
    )
    def test_sub_module_private_completions(self):
        cases = (
            # Return public methods by default
            ("from foo import \t\n", "from foo import public"),
            # Return private methods if explicitly specified
            ("from foo import _\t\n", "from foo import _private"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    def test_builtin_completion_top_level(self):
        cases = (
            ("import bui\t\n", "import builtins"),
            ("from bui\t\n", "from builtins"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    def test_relative_import_completions(self):
        cases = (
            (None, "from .readl\t\n", "from .readl"),
            (None, "from . import readl\t\n", "from . import readl"),
            ("_pyrepl", "from .readl\t\n", "from .readline"),
            ("_pyrepl", "from . import readl\t\n", "from . import readline"),
        )
        for package, code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={"__package__": package})
                output = reader.readline()
                self.assertEqual(output, expected)

    @patch("pkgutil.iter_modules", lambda: [ModuleInfo(None, "valid_name", True),
                                            ModuleInfo(None, "invalid-name", True)])
    def test_invalid_identifiers(self):
        # Make sure modules which are not valid identifiers
        # are not suggested as those cannot be imported via 'import'.
        cases = (
            ("import valid\t\n", "import valid_name"),
            # 'invalid-name' contains a dash and should not be completed
            ("import invalid\t\n", "import invalid"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    def test_no_fallback_on_regular_completion(self):
        cases = (
            ("import pri\t\n", "import pri"),
            ("from pri\t\n", "from pri"),
            ("from typing import Na\t\n", "from typing import Na"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    def test_hardcoded_stdlib_submodules(self):
        cases = (
            ("import collections.\t\n", "import collections.abc"),
            ("from os import \t\n", "from os import path"),
            ("import xml.parsers.expat.\t\te\t\n\n", "import xml.parsers.expat.errors"),
            ("from xml.parsers.expat import \t\tm\t\n\n", "from xml.parsers.expat import model"),
        )
        for code, expected in cases:
            with self.subTest(code=code):
                events = code_to_events(code)
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, expected)

    def test_hardcoded_stdlib_submodules_not_proposed_if_local_import(self):
        with tempfile.TemporaryDirectory() as _dir:
            dir = pathlib.Path(_dir)
            (dir / "collections").mkdir()
            (dir / "collections" / "__init__.py").touch()
            (dir / "collections" / "foo.py").touch()
            with patch.object(sys, "path", [dir, *sys.path]):
                events = code_to_events("import collections.\t\n")
                reader = self.prepare_reader(events, namespace={})
                output = reader.readline()
                self.assertEqual(output, "import collections.foo")

    def test_get_path_and_prefix(self):
        cases = (
            ('', ('', '')),
            ('.', ('.', '')),
            ('..', ('..', '')),
            ('.foo', ('.', 'foo')),
            ('..foo', ('..', 'foo')),
            ('..foo.', ('..foo', '')),
            ('..foo.bar', ('..foo', 'bar')),
            ('.foo.bar.', ('.foo.bar', '')),
            ('..foo.bar.', ('..foo.bar', '')),
            ('foo', ('', 'foo')),
            ('foo.', ('foo', '')),
            ('foo.bar', ('foo', 'bar')),
            ('foo.bar.', ('foo.bar', '')),
            ('foo.bar.baz', ('foo.bar', 'baz')),
        )
        completer = ModuleCompleter()
        for name, expected in cases:
            with self.subTest(name=name):
                self.assertEqual(completer.get_path_and_prefix(name), expected)

    def test_parse(self):
        cases = (
            ('import ', (None, '')),
            ('import foo', (None, 'foo')),
            ('import foo,', (None, '')),
            ('import foo, ', (None, '')),
            ('import foo, bar', (None, 'bar')),
            ('import foo, bar, baz', (None, 'baz')),
            ('import foo as bar,', (None, '')),
            ('import foo as bar, ', (None, '')),
            ('import foo as bar, baz', (None, 'baz')),
            ('import a.', (None, 'a.')),
            ('import a.b', (None, 'a.b')),
            ('import a.b.', (None, 'a.b.')),
            ('import a.b.c', (None, 'a.b.c')),
            ('import a.b.c, foo', (None, 'foo')),
            ('import a.b.c, foo.bar', (None, 'foo.bar')),
            ('import a.b.c, foo.bar,', (None, '')),
            ('import a.b.c, foo.bar, ', (None, '')),
            ('from foo', ('foo', None)),
            ('from a.', ('a.', None)),
            ('from a.b', ('a.b', None)),
            ('from a.b.', ('a.b.', None)),
            ('from a.b.c', ('a.b.c', None)),
            ('from foo import ', ('foo', '')),
            ('from foo import a', ('foo', 'a')),
            ('from ', ('', None)),
            ('from . import a', ('.', 'a')),
            ('from .foo import a', ('.foo', 'a')),
            ('from ..foo import a', ('..foo', 'a')),
            ('from foo import (', ('foo', '')),
            ('from foo import ( ', ('foo', '')),
            ('from foo import (a', ('foo', 'a')),
            ('from foo import (a,', ('foo', '')),
            ('from foo import (a, ', ('foo', '')),
            ('from foo import (a, c', ('foo', 'c')),
            ('from foo import (a as b, c', ('foo', 'c')),
        )
        for code, parsed in cases:
            parser = ImportParser(code)
            actual = parser.parse()
            with self.subTest(code=code):
                self.assertEqual(actual, parsed)
            # The parser should not get tripped up by any
            # other preceding statements
            _code = f'import xyz\n{code}'
            parser = ImportParser(_code)
            actual = parser.parse()
            with self.subTest(code=_code):
                self.assertEqual(actual, parsed)
            _code = f'import xyz;{code}'
            parser = ImportParser(_code)
            actual = parser.parse()
            with self.subTest(code=_code):
                self.assertEqual(actual, parsed)

    def test_parse_error(self):
        cases = (
            '',
            'import foo ',
            'from foo ',
            'import foo. ',
            'import foo.bar ',
            'from foo ',
            'from foo. ',
            'from foo.bar ',
            'from foo import bar ',
            'from foo import (bar ',
            'from foo import bar, baz ',
            'import foo as',
            'import a. as',
            'import a.b as',
            'import a.b. as',
            'import a.b.c as',
            'import (foo',
            'import (',
            'import .foo',
            'import ..foo',
            'import .foo.bar',
            'import foo; x = 1',
            'import a.; x = 1',
            'import a.b; x = 1',
            'import a.b.; x = 1',
            'import a.b.c; x = 1',
            'from foo import a as',
            'from foo import a. as',
            'from foo import a.b as',
            'from foo import a.b. as',
            'from foo import a.b.c as',
            'from foo impo',
            'import import',
            'import from',
            'import as',
            'from import',
            'from from',
            'from as',
            'from foo import import',
            'from foo import from',
            'from foo import as',
        )
        for code in cases:
            parser = ImportParser(code)
            actual = parser.parse()
            with self.subTest(code=code):
                self.assertEqual(actual, None)


class TestHardcodedSubmodules(TestCase):
    def test_hardcoded_stdlib_submodules_are_importable(self):
        for parent_path, submodules in HARDCODED_SUBMODULES.items():
            for module_name in submodules:
                path = f"{parent_path}.{module_name}"
                with self.subTest(path=path):
                    # We can't use importlib.util.find_spec here,
                    # since some hardcoded submodules parents are
                    # not proper packages
                    importlib.import_module(path)


class TestPasteEvent(TestCase):
    def prepare_reader(self, events):
        console = FakeConsole(events)
        config = ReadlineConfig(readline_completer=None)
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    def test_paste(self):
        # fmt: off
        code = (
            "def a():\n"
            "  for x in range(10):\n"
            "    if x%2:\n"
            "      print(x)\n"
            "    else:\n"
            "      pass\n"
        )
        # fmt: on

        events = itertools.chain(
            [
                Event(evt="key", data="f3", raw=bytearray(b"\x1bOR")),
            ],
            code_to_events(code),
            [
                Event(evt="key", data="f3", raw=bytearray(b"\x1bOR")),
            ],
            code_to_events("\n"),
        )
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, code)

    def test_paste_mid_newlines(self):
        # fmt: off
        code = (
            "def f():\n"
            "  x = y\n"
            "  \n"
            "  y = z\n"
        )
        # fmt: on

        events = itertools.chain(
            [
                Event(evt="key", data="f3", raw=bytearray(b"\x1bOR")),
            ],
            code_to_events(code),
            [
                Event(evt="key", data="f3", raw=bytearray(b"\x1bOR")),
            ],
            code_to_events("\n"),
        )
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, code)

    def test_paste_mid_newlines_not_in_paste_mode(self):
        # fmt: off
        code = (
            "def f():\n"
                "x = y\n"
                "\n"
                "y = z\n\n"
        )

        expected = (
            "def f():\n"
            "    x = y\n"
            "    "
        )
        # fmt: on

        events = code_to_events(code)
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, expected)

    def test_paste_not_in_paste_mode(self):
        # fmt: off
        input_code = (
            "def a():\n"
                "for x in range(10):\n"
                    "if x%2:\n"
                        "print(x)\n"
                    "else:\n"
                        "pass\n\n"
        )

        output_code = (
            "def a():\n"
            "    for x in range(10):\n"
            "        if x%2:\n"
            "            print(x)\n"
            "            else:"
        )
        # fmt: on

        events = code_to_events(input_code)
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_bracketed_paste(self):
        """Test that bracketed paste using \x1b[200~ and \x1b[201~ works."""
        # fmt: off
        input_code = (
            "def a():\n"
            "  for x in range(10):\n"
            "\n"
            "    if x%2:\n"
            "      print(x)\n"
            "\n"
            "    else:\n"
            "      pass\n"
        )

        output_code = (
            "def a():\n"
            "  for x in range(10):\n"
            "\n"
            "    if x%2:\n"
            "      print(x)\n"
            "\n"
            "    else:\n"
            "      pass\n"
        )
        # fmt: on

        paste_start = "\x1b[200~"
        paste_end = "\x1b[201~"

        events = itertools.chain(
            code_to_events(paste_start),
            code_to_events(input_code),
            code_to_events(paste_end),
            code_to_events("\n"),
        )
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, output_code)

    def test_bracketed_paste_single_line(self):
        input_code = "oneline"

        paste_start = "\x1b[200~"
        paste_end = "\x1b[201~"

        events = itertools.chain(
            code_to_events(paste_start),
            code_to_events(input_code),
            code_to_events(paste_end),
            code_to_events("\n"),
        )
        reader = self.prepare_reader(events)
        output = multiline_input(reader)
        self.assertEqual(output, input_code)


@skipUnless(pty, "requires pty")
class TestDumbTerminal(ReplTestCase):
    def test_dumb_terminal_exits_cleanly(self):
        env = os.environ.copy()
        env.pop('PYTHON_BASIC_REPL', None)
        env.update({"TERM": "dumb"})
        output, exit_code = self.run_repl("exit()\n", env=env)
        self.assertEqual(exit_code, 0)
        self.assertIn("warning: can't use pyrepl", output)
        self.assertNotIn("Exception", output)
        self.assertNotIn("Traceback", output)


@skipUnless(pty, "requires pty")
@skipIf((os.environ.get("TERM") or "dumb") == "dumb", "can't use pyrepl in dumb terminal")
class TestMain(ReplTestCase):
    def setUp(self):
        # Cleanup from PYTHON* variables to isolate from local
        # user settings, see #121359.  Such variables should be
        # added later in test methods to patched os.environ.
        super().setUp()
        patcher = patch('os.environ', new=make_clean_env())
        self.addCleanup(patcher.stop)
        patcher.start()

    @force_not_colorized
    def test_exposed_globals_in_repl(self):
        pre = "['__builtins__'"
        post = "'__loader__', '__name__', '__package__', '__spec__']"
        output, exit_code = self.run_repl(["sorted(dir())", "exit()"], skip=True)
        self.assertEqual(exit_code, 0)

        # if `__main__` is not a file (impossible with pyrepl)
        case1 = f"{pre}, '__doc__', {post}" in output

        # if `__main__` is an uncached .py file (no .pyc)
        case2 = f"{pre}, '__doc__', '__file__', {post}" in output

        # if `__main__` is a cached .pyc file and the .py source exists
        case3 = f"{pre}, '__cached__', '__doc__', '__file__', {post}" in output

        # if `__main__` is a cached .pyc file but there's no .py source file
        case4 = f"{pre}, '__cached__', '__doc__', {post}" in output

        self.assertTrue(case1 or case2 or case3 or case4, output)

    def _assertMatchOK(
            self, var: str, expected: str | re.Pattern, actual: str
    ) -> None:
        if isinstance(expected, re.Pattern):
            self.assertTrue(
                expected.match(actual),
                f"{var}={actual} does not match {expected.pattern}",
            )
        else:
            self.assertEqual(
                actual,
                expected,
                f"expected {var}={expected}, got {var}={actual}",
            )

    @force_not_colorized
    def _run_repl_globals_test(self, expectations, *, as_file=False, as_module=False, pythonstartup=False):
        clean_env = make_clean_env()
        clean_env["NO_COLOR"] = "1"  # force_not_colorized doesn't touch subprocesses

        with tempfile.TemporaryDirectory() as td:
            blue = pathlib.Path(td) / "blue"
            blue.mkdir()
            mod = blue / "calx.py"
            mod.write_text("FOO = 42", encoding="utf-8")
            startup = blue / "startup.py"
            startup.write_text("BAR = 64", encoding="utf-8")
            commands = [
                "print(f'^{" + var + "=}')" for var in expectations
            ] + ["exit()"]
            if pythonstartup:
                clean_env["PYTHONSTARTUP"] = str(startup)
            if as_file and as_module:
                self.fail("as_file and as_module are mutually exclusive")
            elif as_file:
                output, exit_code = self.run_repl(
                    commands,
                    cmdline_args=[str(mod)],
                    env=clean_env,
                    skip=True,
                )
            elif as_module:
                output, exit_code = self.run_repl(
                    commands,
                    cmdline_args=["-m", "blue.calx"],
                    env=clean_env,
                    cwd=td,
                    skip=True,
                )
            else:
                output, exit_code = self.run_repl(
                    commands,
                    cmdline_args=[],
                    env=clean_env,
                    cwd=td,
                    skip=True,
                )

        self.assertEqual(exit_code, 0)
        for var, expected in expectations.items():
            with self.subTest(var=var, expected=expected):
                if m := re.search(rf"\^{var}=(.+?)[\r\n]", output):
                    self._assertMatchOK(var, expected, actual=m.group(1))
                else:
                    self.fail(f"{var}= not found in output: {output!r}\n\n{output}")

        self.assertNotIn("Exception", output)
        self.assertNotIn("Traceback", output)

    def test_globals_initialized_as_default(self):
        expectations = {
            "__name__": "'__main__'",
            "__package__": "None",
            # "__file__" is missing in -i, like in the basic REPL
        }
        self._run_repl_globals_test(expectations)

    def test_globals_initialized_from_pythonstartup(self):
        expectations = {
            "BAR": "64",
            "__name__": "'__main__'",
            "__package__": "None",
            # "__file__" is missing in -i, like in the basic REPL
        }
        self._run_repl_globals_test(expectations, pythonstartup=True)

    def test_inspect_keeps_globals_from_inspected_file(self):
        expectations = {
            "FOO": "42",
            "__name__": "'__main__'",
            "__package__": "None",
            # "__file__" is missing in -i, like in the basic REPL
        }
        self._run_repl_globals_test(expectations, as_file=True)

    def test_inspect_keeps_globals_from_inspected_file_with_pythonstartup(self):
        expectations = {
            "FOO": "42",
            "BAR": "64",
            "__name__": "'__main__'",
            "__package__": "None",
            # "__file__" is missing in -i, like in the basic REPL
        }
        self._run_repl_globals_test(expectations, as_file=True, pythonstartup=True)

    def test_inspect_keeps_globals_from_inspected_module(self):
        expectations = {
            "FOO": "42",
            "__name__": "'__main__'",
            "__package__": "'blue'",
            "__file__": re.compile(r"^'.*calx.py'$"),
        }
        self._run_repl_globals_test(expectations, as_module=True)

    def test_inspect_keeps_globals_from_inspected_module_with_pythonstartup(self):
        expectations = {
            "FOO": "42",
            "BAR": "64",
            "__name__": "'__main__'",
            "__package__": "'blue'",
            "__file__": re.compile(r"^'.*calx.py'$"),
        }
        self._run_repl_globals_test(expectations, as_module=True, pythonstartup=True)

    @force_not_colorized
    def test_python_basic_repl(self):
        env = os.environ.copy()
        pyrepl_commands = "clear\nexit()\n"
        env.pop("PYTHON_BASIC_REPL", None)
        output, exit_code = self.run_repl(pyrepl_commands, env=env, skip=True)
        self.assertEqual(exit_code, 0)
        self.assertNotIn("Exception", output)
        self.assertNotIn("NameError", output)
        self.assertNotIn("Traceback", output)

        basic_commands = "help\nexit()\n"
        env["PYTHON_BASIC_REPL"] = "1"
        output, exit_code = self.run_repl(basic_commands, env=env)
        self.assertEqual(exit_code, 0)
        self.assertIn("Type help() for interactive help", output)
        self.assertNotIn("Exception", output)
        self.assertNotIn("Traceback", output)

        # The site module must not load _pyrepl if PYTHON_BASIC_REPL is set
        commands = ("import sys\n"
                    "print('_pyrepl' in sys.modules)\n"
                    "exit()\n")
        env["PYTHON_BASIC_REPL"] = "1"
        output, exit_code = self.run_repl(commands, env=env)
        self.assertEqual(exit_code, 0)
        self.assertIn("False", output)
        self.assertNotIn("True", output)
        self.assertNotIn("Exception", output)
        self.assertNotIn("Traceback", output)

    @force_not_colorized
    def test_no_pyrepl_source_in_exc(self):
        # Avoid using _pyrepl/__main__.py in traceback reports
        # See https://github.com/python/cpython/issues/129098.
        pyrepl_main_file = os.path.join(STDLIB_DIR, "_pyrepl", "__main__.py")
        self.assertTrue(os.path.exists(pyrepl_main_file), pyrepl_main_file)
        with open(pyrepl_main_file) as fp:
            excluded_lines = fp.readlines()
        excluded_lines = list(filter(None, map(str.strip, excluded_lines)))

        for filename in ['?', 'unknown-filename', '<foo>', '<...>']:
            self._test_no_pyrepl_source_in_exc(filename, excluded_lines)

    def _test_no_pyrepl_source_in_exc(self, filename, excluded_lines):
        with EnvironmentVarGuard() as env, self.subTest(filename=filename):
            env.unset("PYTHON_BASIC_REPL")
            commands = (f"eval(compile('spam', {filename!r}, 'eval'))\n"
                        f"exit()\n")
            output, _ = self.run_repl(commands, env=env)
            self.assertIn("Traceback (most recent call last)", output)
            self.assertIn("NameError: name 'spam' is not defined", output)
            for line in excluded_lines:
                with self.subTest(line=line):
                    self.assertNotIn(line, output)

    @force_not_colorized
    def test_bad_sys_excepthook_doesnt_crash_pyrepl(self):
        env = os.environ.copy()
        commands = ("import sys\n"
                    "sys.excepthook = 1\n"
                    "1/0\n"
                    "exit()\n")

        def check(output, exitcode):
            self.assertIn("Error in sys.excepthook:", output)
            self.assertEqual(output.count("'int' object is not callable"), 1)
            self.assertIn("Original exception was:", output)
            self.assertIn("division by zero", output)
            self.assertEqual(exitcode, 0)
        env.pop("PYTHON_BASIC_REPL", None)
        output, exit_code = self.run_repl(commands, env=env, skip=True)
        check(output, exit_code)

        env["PYTHON_BASIC_REPL"] = "1"
        output, exit_code = self.run_repl(commands, env=env)
        check(output, exit_code)

    def test_not_wiping_history_file(self):
        # skip, if readline module is not available
        import_module('readline')

        hfile = tempfile.NamedTemporaryFile(delete=False)
        self.addCleanup(unlink, hfile.name)
        env = os.environ.copy()
        env["PYTHON_HISTORY"] = hfile.name
        commands = "123\nspam\nexit()\n"

        env.pop("PYTHON_BASIC_REPL", None)
        output, exit_code = self.run_repl(commands, env=env)
        self.assertEqual(exit_code, 0)
        self.assertIn("123", output)
        self.assertIn("spam", output)
        self.assertNotEqual(pathlib.Path(hfile.name).stat().st_size, 0)

        hfile.file.truncate()
        hfile.close()

        env["PYTHON_BASIC_REPL"] = "1"
        output, exit_code = self.run_repl(commands, env=env)
        self.assertEqual(exit_code, 0)
        self.assertIn("123", output)
        self.assertIn("spam", output)
        self.assertNotEqual(pathlib.Path(hfile.name).stat().st_size, 0)

    @force_not_colorized
    def test_correct_filename_in_syntaxerrors(self):
        env = os.environ.copy()
        commands = "a b c\nexit()\n"
        output, exit_code = self.run_repl(commands, env=env, skip=True)
        self.assertIn("SyntaxError: invalid syntax", output)
        self.assertIn("<python-input-0>", output)
        commands = " b\nexit()\n"
        output, exit_code = self.run_repl(commands, env=env)
        self.assertIn("IndentationError: unexpected indent", output)
        self.assertIn("<python-input-0>", output)

    @force_not_colorized
    def test_proper_tracebacklimit(self):
        env = os.environ.copy()
        for set_tracebacklimit in [True, False]:
            commands = ("import sys\n" +
                        ("sys.tracebacklimit = 1\n" if set_tracebacklimit else "") +
                        "def x1(): 1/0\n\n"
                        "def x2(): x1()\n\n"
                        "def x3(): x2()\n\n"
                        "x3()\n"
                        "exit()\n")

            for basic_repl in [True, False]:
                if basic_repl:
                    env["PYTHON_BASIC_REPL"] = "1"
                else:
                    env.pop("PYTHON_BASIC_REPL", None)
                with self.subTest(set_tracebacklimit=set_tracebacklimit,
                                  basic_repl=basic_repl):
                    output, exit_code = self.run_repl(commands, env=env, skip=True)
                    self.assertIn("in x1", output)
                    if set_tracebacklimit:
                        self.assertNotIn("in x2", output)
                        self.assertNotIn("in x3", output)
                        self.assertNotIn("in <module>", output)
                    else:
                        self.assertIn("in x2", output)
                        self.assertIn("in x3", output)
                        self.assertIn("in <module>", output)

    def test_null_byte(self):
        output, exit_code = self.run_repl("\x00\nexit()\n")
        self.assertEqual(exit_code, 0)
        self.assertNotIn("TypeError", output)

    @force_not_colorized
    def test_non_string_suggestion_candidates(self):
        commands = ("import runpy\n"
                    "runpy._run_module_code('blech', {0: '', 'bluch': ''}, '')\n"
                    "exit()\n")

        output, exit_code = self.run_repl(commands)
        self.assertEqual(exit_code, 0)
        self.assertNotIn("all elements in 'candidates' must be strings", output)
        self.assertIn("bluch", output)

    def test_readline_history_file(self):
        # skip, if readline module is not available
        readline = import_module('readline')
        if readline.backend != "editline":
            self.skipTest("GNU readline is not affected by this issue")

        with tempfile.NamedTemporaryFile() as hfile:
            env = os.environ.copy()
            env["PYTHON_HISTORY"] = hfile.name

            env["PYTHON_BASIC_REPL"] = "1"
            output, exit_code = self.run_repl("spam \nexit()\n", env=env)
            self.assertEqual(exit_code, 0)
            self.assertIn("spam ", output)
            self.assertNotEqual(pathlib.Path(hfile.name).stat().st_size, 0)
            self.assertIn("spam\\040", pathlib.Path(hfile.name).read_text())

            env.pop("PYTHON_BASIC_REPL", None)
            output, exit_code = self.run_repl("exit\n", env=env)
            self.assertEqual(exit_code, 0)
            self.assertNotIn("\\040", pathlib.Path(hfile.name).read_text())

    def test_history_survive_crash(self):
        env = os.environ.copy()

        with tempfile.NamedTemporaryFile() as hfile:
            env["PYTHON_HISTORY"] = hfile.name

            commands = "1\n2\n3\nexit()\n"
            output, exit_code = self.run_repl(commands, env=env, skip=True)
            self.assertEqual(exit_code, 0)

            # Run until "0xcafe" is printed (as "51966") and then kill the
            # process to simulate a crash. Note that the output also includes
            # the echoed input commands.
            commands = "spam\nimport time\n0xcafe\ntime.sleep(1000)\nquit\n"
            output, exit_code = self.run_repl(commands, env=env,
                                              exit_on_output="51966")
            self.assertNotEqual(exit_code, 0)

            history = pathlib.Path(hfile.name).read_text()
            self.assertIn("2", history)
            self.assertIn("exit()", history)
            self.assertIn("spam", history)
            self.assertIn("import time", history)
            # History is written after each command's output is printed to the
            # console, so depending on how quickly the process is killed,
            # the last command may or may not be written to the history file.
            self.assertNotIn("sleep", history)
            self.assertNotIn("quit", history)

    def test_keyboard_interrupt_after_isearch(self):
        output, exit_code = self.run_repl("\x12\x03exit\n")
        self.assertEqual(exit_code, 0)

    def test_prompt_after_help(self):
        output, exit_code = self.run_repl(["help", "q", "exit"])

        # Regex pattern to remove ANSI escape sequences
        ansi_escape = re.compile(r"(\x1B(=|>|(\[)[0-?]*[ -\/]*[@-~]))")
        cleaned_output = ansi_escape.sub("", output)
        self.assertEqual(exit_code, 0)

        # Ensure that we don't see multiple prompts after exiting `help`
        # Extra stuff (newline and `exit` rewrites) are necessary
        # because of how run_repl works.
        self.assertNotIn(">>> \n>>> >>>", cleaned_output)

    @skipUnless(Py_DEBUG, '-X showrefcount requires a Python debug build')
    def test_showrefcount(self):
        env = os.environ.copy()
        env.pop("PYTHON_BASIC_REPL", "")
        output, _ = self.run_repl("1\n1+2\nexit()\n", cmdline_args=['-Xshowrefcount'], env=env)
        matches = re.findall(r'\[-?\d+ refs, \d+ blocks\]', output)
        self.assertEqual(len(matches), 3)

        env["PYTHON_BASIC_REPL"] = "1"
        output, _ = self.run_repl("1\n1+2\nexit()\n", cmdline_args=['-Xshowrefcount'], env=env)
        matches = re.findall(r'\[-?\d+ refs, \d+ blocks\]', output)
        self.assertEqual(len(matches), 3)

    def test_detect_pip_usage_in_repl(self):
        for pip_cmd in ("pip", "pip3", "python -m pip", "python3 -m pip"):
            with self.subTest(pip_cmd=pip_cmd):
                output, exit_code = self.run_repl([f"{pip_cmd} install sampleproject", "exit"])
                self.assertIn("SyntaxError", output)
                hint = (
                    "The Python package manager (pip) can only be used"
                    " outside of the Python REPL"
                )
                self.assertIn(hint, output)

class TestPyReplCtrlD(TestCase):
    """Test Ctrl+D behavior in _pyrepl to match old pre-3.13 REPL behavior.

    Ctrl+D should:
    - Exit on empty buffer (raises EOFError)
    - Delete character when cursor is in middle of line
    - Perform no operation when cursor is at end of line without newline
    - Exit multiline mode when cursor is at end with trailing newline
    - Run code up to that point when pressed on blank line with preceding lines
    """
    def prepare_reader(self, events):
        console = FakeConsole(events)
        config = ReadlineConfig(readline_completer=None)
        reader = ReadlineAlikeReader(console=console, config=config)
        return reader

    def test_ctrl_d_empty_line(self):
        """Test that pressing Ctrl+D on empty line exits the program"""
        events = [
            Event(evt="key", data="\x04", raw=bytearray(b"\x04")),  # Ctrl+D
        ]
        reader = self.prepare_reader(events)
        with self.assertRaises(EOFError):
            multiline_input(reader)

    def test_ctrl_d_multiline_with_new_line(self):
        """Test that pressing Ctrl+D in multiline mode with trailing newline exits multiline mode"""
        events = itertools.chain(
            code_to_events("def f():\n    pass\n"),  # Enter multiline mode with trailing newline
            [
                Event(evt="key", data="\x04", raw=bytearray(b"\x04")),  # Ctrl+D
            ],
        )
        reader, _ = handle_all_events(events)
        self.assertTrue(reader.finished)
        self.assertEqual("def f():\n    pass\n", "".join(reader.buffer))

    def test_ctrl_d_multiline_middle_of_line(self):
        """Test that pressing Ctrl+D in multiline mode with cursor in middle deletes character"""
        events = itertools.chain(
            code_to_events("def f():\n    hello world"),  # Enter multiline mode
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))
            ] * 5,  # move cursor to 'w' in "world"
            [
                Event(evt="key", data="\x04", raw=bytearray(b"\x04"))
            ], # Ctrl+D should delete 'w'
        )
        reader, _ = handle_all_events(events)
        self.assertFalse(reader.finished)
        self.assertEqual("def f():\n    hello orld", "".join(reader.buffer))

    def test_ctrl_d_multiline_end_of_line_no_newline(self):
        """Test that pressing Ctrl+D at end of line without newline performs no operation"""
        events = itertools.chain(
            code_to_events("def f():\n    hello"),  # Enter multiline mode, no trailing newline
            [
                Event(evt="key", data="\x04", raw=bytearray(b"\x04"))
            ],  # Ctrl+D should be no-op
        )
        reader, _ = handle_all_events(events)
        self.assertFalse(reader.finished)
        self.assertEqual("def f():\n    hello", "".join(reader.buffer))

    def test_ctrl_d_single_line_middle_of_line(self):
        """Test that pressing Ctrl+D in single line mode deletes current character"""
        events = itertools.chain(
            code_to_events("hello"),
            [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],  # move left
            [Event(evt="key", data="\x04", raw=bytearray(b"\x04"))],    # Ctrl+D
        )
        reader, _ = handle_all_events(events)
        self.assertEqual("hell", "".join(reader.buffer))

    def test_ctrl_d_single_line_end_no_newline(self):
        """Test that pressing Ctrl+D at end of single line without newline does nothing"""
        events = itertools.chain(
            code_to_events("hello"),  # cursor at end of line
            [Event(evt="key", data="\x04", raw=bytearray(b"\x04"))],  # Ctrl+D
        )
        reader, _ = handle_all_events(events)
        self.assertEqual("hello", "".join(reader.buffer))

### File: test_reader.py

In [ ]:
import itertools
import functools
import rlcompleter
from textwrap import dedent
from unittest import TestCase
from unittest.mock import MagicMock
from test.support import force_colorized_test_class, force_not_colorized_test_class

from .support import handle_all_events, handle_events_narrow_console
from .support import ScreenEqualMixin, code_to_events
from .support import prepare_reader, prepare_console
from _pyrepl.console import Event
from _pyrepl.reader import Reader
from _colorize import default_theme


overrides = {"reset": "z", "soft_keyword": "K"}
colors = {overrides.get(k, k[0].lower()): v for k, v in default_theme.syntax.items()}


@force_not_colorized_test_class
class TestReader(ScreenEqualMixin, TestCase):
    def test_calc_screen_wrap_simple(self):
        events = code_to_events(10 * "a")
        reader, _ = handle_events_narrow_console(events)
        self.assert_screen_equal(reader, f"{9*"a"}\\\na")

    def test_calc_screen_wrap_wide_characters(self):
        events = code_to_events(8 * "a" + "樂")
        reader, _ = handle_events_narrow_console(events)
        self.assert_screen_equal(reader, f"{8*"a"}\\\n樂")

    def test_calc_screen_wrap_three_lines(self):
        events = code_to_events(20 * "a")
        reader, _ = handle_events_narrow_console(events)
        self.assert_screen_equal(reader, f"{9*"a"}\\\n{9*"a"}\\\naa")

    def test_calc_screen_prompt_handling(self):
        def prepare_reader_keep_prompts(*args, **kwargs):
            reader = prepare_reader(*args, **kwargs)
            del reader.get_prompt
            reader.ps1 = ">>> "
            reader.ps2 = ">>> "
            reader.ps3 = "... "
            reader.ps4 = ""
            reader.can_colorize = False
            reader.paste_mode = False
            return reader

        events = code_to_events("if some_condition:\nsome_function()")
        reader, _ = handle_events_narrow_console(
            events,
            prepare_reader=prepare_reader_keep_prompts,
        )
        # fmt: off
        self.assert_screen_equal(
            reader,
            (
            ">>> if so\\\n"
            "me_condit\\\n"
            "ion:\n"
            "...     s\\\n"
            "ome_funct\\\n"
            "ion()"
            )
        )
        # fmt: on

    def test_calc_screen_wrap_three_lines_mixed_character(self):
        # fmt: off
        code = (
            "def f():\n"
           f"  {8*"a"}\n"
           f"  {5*"樂"}"
        )
        # fmt: on

        events = code_to_events(code)
        reader, _ = handle_events_narrow_console(events)

        # fmt: off
        self.assert_screen_equal(
            reader,
            (
                "def f():\n"
               f"  {7*"a"}\\\n"
                "a\n"
               f"  {3*"樂"}\\\n"
                "樂樂"
            ),
            clean=True,
        )
        # fmt: on

    def test_calc_screen_backspace(self):
        events = itertools.chain(
            code_to_events("aaa"),
            [
                Event(evt="key", data="backspace", raw=bytearray(b"\x7f")),
            ],
        )
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, "aa")

    def test_calc_screen_wrap_removes_after_backspace(self):
        events = itertools.chain(
            code_to_events(10 * "a"),
            [
                Event(evt="key", data="backspace", raw=bytearray(b"\x7f")),
            ],
        )
        reader, _ = handle_events_narrow_console(events)
        self.assert_screen_equal(reader, 9 * "a")

    def test_calc_screen_backspace_in_second_line_after_wrap(self):
        events = itertools.chain(
            code_to_events(11 * "a"),
            [
                Event(evt="key", data="backspace", raw=bytearray(b"\x7f")),
            ],
        )
        reader, _ = handle_events_narrow_console(events)
        self.assert_screen_equal(reader, f"{9*"a"}\\\na")

    def test_setpos_for_xy_simple(self):
        events = code_to_events("11+11")
        reader, _ = handle_all_events(events)
        reader.setpos_from_xy(0, 0)
        self.assertEqual(reader.pos, 0)

    def test_setpos_from_xy_multiple_lines(self):
        # fmt: off
        code = (
            "def foo():\n"
            "  return 1"
        )
        # fmt: on

        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        reader.setpos_from_xy(2, 1)
        self.assertEqual(reader.pos, 13)

    def test_setpos_from_xy_after_wrap(self):
        # fmt: off
        code = (
            "def foo():\n"
            "  hello"
        )
        # fmt: on

        events = code_to_events(code)
        reader, _ = handle_events_narrow_console(events)
        reader.setpos_from_xy(2, 2)
        self.assertEqual(reader.pos, 13)

    def test_setpos_fromxy_in_wrapped_line(self):
        # fmt: off
        code = (
            "def foo():\n"
            "  hello"
        )
        # fmt: on

        events = code_to_events(code)
        reader, _ = handle_events_narrow_console(events)
        reader.setpos_from_xy(0, 1)
        self.assertEqual(reader.pos, 9)

    def test_up_arrow_after_ctrl_r(self):
        events = iter(
            [
                Event(evt="key", data="\x12", raw=bytearray(b"\x12")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
            ]
        )

        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, "")

    def test_newline_within_block_trailing_whitespace(self):
        # fmt: off
        code = (
            "def foo():\n"
                 "a = 1\n"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                # go to the end of the first line
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="\x05", raw=bytearray(b"\x1bO5")),
                # new lines in-block shouldn't terminate the block
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                # end of line 2
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="key", data="\x05", raw=bytearray(b"\x1bO5")),
                # a double new line in-block should terminate the block
                # even if its followed by whitespace
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
                Event(evt="key", data="\n", raw=bytearray(b"\n")),
            ],
        )

        no_paste_reader = functools.partial(prepare_reader, paste_mode=False)
        reader, _ = handle_all_events(events, prepare_reader=no_paste_reader)

        expected = (
            "def foo():\n"
            "    \n"
            "    \n"
            "    a = 1\n"
            "    \n"
            "    "  # HistoricalReader will trim trailing whitespace
        )
        self.assert_screen_equal(reader, expected, clean=True)
        self.assertTrue(reader.finished)

    def test_input_hook_is_called_if_set(self):
        input_hook = MagicMock()

        def _prepare_console(events):
            console = MagicMock()
            console.get_event.side_effect = events
            console.height = 100
            console.width = 80
            console.input_hook = input_hook
            return console

        events = code_to_events("a")
        reader, _ = handle_all_events(events, prepare_console=_prepare_console)

        self.assertEqual(len(input_hook.mock_calls), 4)

    def test_keyboard_interrupt_clears_screen(self):
        namespace = {"itertools": itertools}
        code = "import itertools\nitertools."
        events = itertools.chain(
            code_to_events(code),
            [
                # Two tabs for completion
                Event(evt="key", data="\t", raw=bytearray(b"\t")),
                Event(evt="key", data="\t", raw=bytearray(b"\t")),
                Event(evt="key", data="\x03", raw=bytearray(b"\x03")),  # Ctrl-C
            ],
        )
        console = prepare_console(events)
        reader = prepare_reader(
            console,
            readline_completer=rlcompleter.Completer(namespace).complete,
        )
        try:
            # we're not using handle_all_events() here to be able to
            # follow the KeyboardInterrupt sequence of events. Normally this
            # happens in simple_interact.run_multiline_interactive_console.
            while True:
                reader.handle1()
        except KeyboardInterrupt:
            # at this point the completions are still visible
            self.assertTrue(len(reader.screen) > 2)
            reader.refresh()
            # after the refresh, they are gone
            self.assertEqual(len(reader.screen), 2)
            self.assert_screen_equal(reader, code, clean=True)
        else:
            self.fail("KeyboardInterrupt not raised.")

    def test_prompt_length(self):
        # Handles simple ASCII prompt
        ps1 = ">>> "
        prompt, l = Reader.process_prompt(ps1)
        self.assertEqual(prompt, ps1)
        self.assertEqual(l, 4)

        # Handles ANSI escape sequences
        ps1 = "\033[0;32m>>> \033[0m"
        prompt, l = Reader.process_prompt(ps1)
        self.assertEqual(prompt, "\033[0;32m>>> \033[0m")
        self.assertEqual(l, 4)

        # Handles ANSI escape sequences bracketed in \001 .. \002
        ps1 = "\001\033[0;32m\002>>> \001\033[0m\002"
        prompt, l = Reader.process_prompt(ps1)
        self.assertEqual(prompt, "\033[0;32m>>> \033[0m")
        self.assertEqual(l, 4)

        # Handles wide characters in prompt
        ps1 = "樂>> "
        prompt, l = Reader.process_prompt(ps1)
        self.assertEqual(prompt, ps1)
        self.assertEqual(l, 5)

        # Handles wide characters AND ANSI sequences together
        ps1 = "\001\033[0;32m\002樂>\001\033[0m\002> "
        prompt, l = Reader.process_prompt(ps1)
        self.assertEqual(prompt, "\033[0;32m樂>\033[0m> ")
        self.assertEqual(l, 5)

    def test_completions_updated_on_key_press(self):
        namespace = {"itertools": itertools}
        code = "itertools."
        events = itertools.chain(
            code_to_events(code),
            [
                # Two tabs for completion
                Event(evt="key", data="\t", raw=bytearray(b"\t")),
                Event(evt="key", data="\t", raw=bytearray(b"\t")),
            ],
            code_to_events("a"),
        )

        completing_reader = functools.partial(
            prepare_reader,
            readline_completer=rlcompleter.Completer(namespace).complete,
        )
        reader, _ = handle_all_events(events, prepare_reader=completing_reader)

        actual = reader.screen
        self.assertEqual(len(actual), 2)
        self.assertEqual(actual[0], f"{code}a")
        self.assertEqual(actual[1].rstrip(), "itertools.accumulate(")

    def test_key_press_on_tab_press_once(self):
        namespace = {"itertools": itertools}
        code = "itertools."
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="\t", raw=bytearray(b"\t")),
            ],
            code_to_events("a"),
        )

        completing_reader = functools.partial(
            prepare_reader,
            readline_completer=rlcompleter.Completer(namespace).complete,
        )
        reader, _ = handle_all_events(events, prepare_reader=completing_reader)

        self.assert_screen_equal(reader, f"{code}a")

    def test_pos2xy_with_no_columns(self):
        console = prepare_console([])
        reader = prepare_reader(console)
        # Simulate a resize to 0 columns
        reader.screeninfo = []
        self.assertEqual(reader.pos2xy(), (0, 0))

    def test_setpos_from_xy_for_non_printing_char(self):
        code = "# non \u200c printing character"
        events = code_to_events(code)

        reader, _ = handle_all_events(events)
        reader.setpos_from_xy(8, 0)
        self.assertEqual(reader.pos, 7)

@force_colorized_test_class
class TestReaderInColor(ScreenEqualMixin, TestCase):
    def test_syntax_highlighting_basic(self):
        code = dedent(
            """\
            import re, sys
            def funct(case: str = sys.platform) -> None:
                match = re.search(
                    "(me)",
                    '''
                    Come on
                      Come on now
                        You know that it's time to emerge
                    ''',
                )
                match case:
                    case "emscripten": print("on the web")
                    case "ios" | "android":
                        print("on the phone")
                    case _: print('arms around', match.group(1))
            type type = type[type]
            """
        )
        expected = dedent(
            """\
            {k}import{z} re{o},{z} sys
            {a}{k}def{z} {d}funct{z}{o}({z}case{o}:{z} {b}str{z} {o}={z} sys{o}.{z}platform{o}){z} {o}->{z} {k}None{z}{o}:{z}
                match {o}={z} re{o}.{z}search{o}({z}
                    {s}"(me)"{z}{o},{z}
                    {s}'''{z}
            {s}        Come on{z}
            {s}          Come on now{z}
            {s}            You know that it's time to emerge{z}
            {s}        '''{z}{o},{z}
                {o}){z}
                {K}match{z} case{o}:{z}
                    {K}case{z} {s}"emscripten"{z}{o}:{z} {b}print{z}{o}({z}{s}"on the web"{z}{o}){z}
                    {K}case{z} {s}"ios"{z} {o}|{z} {s}"android"{z}{o}:{z}
                        {b}print{z}{o}({z}{s}"on the phone"{z}{o}){z}
                    {K}case{z} {K}_{z}{o}:{z} {b}print{z}{o}({z}{s}'arms around'{z}{o},{z} match{o}.{z}group{o}({z}{n}1{z}{o}){z}{o}){z}
            {K}type{z} {b}type{z} {o}={z} {b}type{z}{o}[{z}{b}type{z}{o}]{z}
            """
        )
        expected_sync = expected.format(a="", **colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected_sync)
        self.assertEqual(reader.pos, 419)
        self.assertEqual(reader.cxy, (0, 16))

        async_msg = "{k}async{z} ".format(**colors)
        expected_async = expected.format(a=async_msg, **colors)
        more_events = itertools.chain(
            code_to_events(code),
            [Event(evt="key", data="up", raw=bytearray(b"\x1bOA"))] * 15,
            code_to_events("async "),
        )
        reader, _ = handle_all_events(more_events)
        self.assert_screen_equal(reader, expected_async)
        self.assertEqual(reader.pos, 21)
        self.assertEqual(reader.cxy, (6, 1))

    def test_syntax_highlighting_incomplete_string_first_line(self):
        code = dedent(
            """\
            def unfinished_function(arg: str = "still typing
            """
        )
        expected = dedent(
            """\
            {k}def{z} {d}unfinished_function{z}{o}({z}arg{o}:{z} {b}str{z} {o}={z} {s}"still typing{z}
            """
        ).format(**colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected)

    def test_syntax_highlighting_incomplete_string_another_line(self):
        code = dedent(
            """\
            def unfinished_function(
                arg: str = "still typing
            """
        )
        expected = dedent(
            """\
            {k}def{z} {d}unfinished_function{z}{o}({z}
                arg{o}:{z} {b}str{z} {o}={z} {s}"still typing{z}
            """
        ).format(**colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected)

    def test_syntax_highlighting_incomplete_multiline_string(self):
        code = dedent(
            """\
            def unfinished_function():
                '''Still writing
                the docstring
            """
        )
        expected = dedent(
            """\
            {k}def{z} {d}unfinished_function{z}{o}({z}{o}){z}{o}:{z}
                {s}'''Still writing{z}
            {s}    the docstring{z}
            """
        ).format(**colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected)

    def test_syntax_highlighting_incomplete_fstring(self):
        code = dedent(
            """\
            def unfinished_function():
                var = f"Single-quote but {
                1
                +
                1
                } multi-line!
            """
        )
        expected = dedent(
            """\
            {k}def{z} {d}unfinished_function{z}{o}({z}{o}){z}{o}:{z}
                var {o}={z} {s}f"{z}{s}Single-quote but {z}{o}{OB}{z}
                {n}1{z}
                {o}+{z}
                {n}1{z}
                {o}{CB}{z}{s} multi-line!{z}
            """
        ).format(OB="{", CB="}", **colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected)

    def test_syntax_highlighting_indentation_error(self):
        code = dedent(
            """\
            def unfinished_function():
                var = 1
               oops
            """
        )
        expected = dedent(
            """\
            {k}def{z} {d}unfinished_function{z}{o}({z}{o}){z}{o}:{z}
                var {o}={z} {n}1{z}
               oops
            """
        ).format(**colors)
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.assert_screen_equal(reader, expected)

    def test_syntax_highlighting_literal_brace_in_fstring_or_tstring(self):
        code = dedent(
            """\
            f"{{"
            f"}}"
            f"a{{b"
            f"a}}b"
            f"a{{b}}c"
            t"a{{b}}c"
            f"{{{0}}}"
            f"{ {0} }"
            """
        )
        expected = dedent(
            """\
            {s}f"{z}{s}<<{z}{s}"{z}
            {s}f"{z}{s}>>{z}{s}"{z}
            {s}f"{z}{s}a<<{z}{s}b{z}{s}"{z}
            {s}f"{z}{s}a>>{z}{s}b{z}{s}"{z}
            {s}f"{z}{s}a<<{z}{s}b>>{z}{s}c{z}{s}"{z}
            {s}t"{z}{s}a<<{z}{s}b>>{z}{s}c{z}{s}"{z}
            {s}f"{z}{s}<<{z}{o}<{z}{n}0{z}{o}>{z}{s}>>{z}{s}"{z}
            {s}f"{z}{o}<{z} {o}<{z}{n}0{z}{o}>{z} {o}>{z}{s}"{z}
            """
        ).format(**colors).replace("<", "{").replace(">", "}")
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, code, clean=True)
        self.maxDiff=None
        self.assert_screen_equal(reader, expected)

    def test_control_characters(self):
        code = 'flag = "🏳️‍🌈"'
        events = code_to_events(code)
        reader, _ = handle_all_events(events)
        self.assert_screen_equal(reader, 'flag = "🏳️\\u200d🌈"', clean=True)
        self.assert_screen_equal(reader, 'flag {o}={z} {s}"🏳️\\u200d🌈"{z}'.format(**colors))

### File: test_terminfo.py

In [ ]:
"""Tests comparing PyREPL's pure Python curses implementation with the standard curses module."""

import json
import os
import subprocess
import sys
import unittest
from test.support import requires, has_subprocess_support
from textwrap import dedent

# Only run these tests if curses is available
requires("curses")

try:
    import _curses
except ImportError:
    try:
        import curses as _curses
    except ImportError:
        _curses = None

from _pyrepl import terminfo


ABSENT_STRING = terminfo.ABSENT_STRING
CANCELLED_STRING = terminfo.CANCELLED_STRING


class TestCursesCompatibility(unittest.TestCase):
    """Test that PyREPL's curses implementation matches the standard curses behavior.

    Python's `curses` doesn't allow calling `setupterm()` again with a different
    $TERM in the same process, so we subprocess all `curses` tests to get correctly
    set up terminfo."""

    @classmethod
    def setUpClass(cls):
        if _curses is None:
            raise unittest.SkipTest(
                "`curses` capability provided to regrtest but `_curses` not importable"
            )

        if not has_subprocess_support:
            raise unittest.SkipTest("test module requires subprocess")

        # we need to ensure there's a terminfo database on the system and that
        # `infocmp` works
        cls.infocmp("dumb")

    def setUp(self):
        self.original_term = os.environ.get("TERM", None)

    def tearDown(self):
        if self.original_term is not None:
            os.environ["TERM"] = self.original_term
        elif "TERM" in os.environ:
            del os.environ["TERM"]

    @classmethod
    def infocmp(cls, term) -> list[str]:
        all_caps = []
        try:
            result = subprocess.run(
                ["infocmp", "-l1", term],
                capture_output=True,
                text=True,
                check=True,
            )
        except Exception:
            raise unittest.SkipTest("calling `infocmp` failed on the system")

        for line in result.stdout.splitlines():
            line = line.strip()
            if line.startswith("#"):
                if "terminfo" not in line and "termcap" in line:
                    # PyREPL terminfo doesn't parse termcap databases
                    raise unittest.SkipTest(
                        "curses using termcap.db: no terminfo database on"
                        " the system"
                    )
            elif "=" in line:
                cap_name = line.split("=")[0]
                all_caps.append(cap_name)

        return all_caps

    def test_setupterm_basic(self):
        """Test basic setupterm functionality."""
        # Test with explicit terminal type
        test_terms = ["xterm", "xterm-256color", "vt100", "ansi"]

        for term in test_terms:
            with self.subTest(term=term):
                ncurses_code = dedent(
                    f"""
                    import _curses
                    import json
                    try:
                        _curses.setupterm({repr(term)}, 1)
                        print(json.dumps({{"success": True}}))
                    except Exception as e:
                        print(json.dumps({{"success": False, "error": str(e)}}))
                    """
                )

                result = subprocess.run(
                    [sys.executable, "-c", ncurses_code],
                    capture_output=True,
                    text=True,
                )
                ncurses_data = json.loads(result.stdout)
                std_success = ncurses_data["success"]

                # Set up with PyREPL curses
                try:
                    terminfo.TermInfo(term, fallback=False)
                    pyrepl_success = True
                except Exception as e:
                    pyrepl_success = False
                    pyrepl_error = e

                # Both should succeed or both should fail
                if std_success:
                    self.assertTrue(
                        pyrepl_success,
                        f"Standard curses succeeded but PyREPL failed for {term}",
                    )
                else:
                    # If standard curses failed, PyREPL might still succeed with fallback
                    # This is acceptable as PyREPL has hardcoded fallbacks
                    pass

    def test_setupterm_none(self):
        """Test setupterm with None (uses TERM from environment)."""
        # Test with current TERM
        ncurses_code = dedent(
            """
            import _curses
            import json
            try:
                _curses.setupterm(None, 1)
                print(json.dumps({"success": True}))
            except Exception as e:
                print(json.dumps({"success": False, "error": str(e)}))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        ncurses_data = json.loads(result.stdout)
        std_success = ncurses_data["success"]

        try:
            terminfo.TermInfo(None, fallback=False)
            pyrepl_success = True
        except Exception:
            pyrepl_success = False

        # Both should have same result
        if std_success:
            self.assertTrue(
                pyrepl_success,
                "Standard curses succeeded but PyREPL failed for None",
            )

    def test_tigetstr_common_capabilities(self):
        """Test tigetstr for common terminal capabilities."""
        # Test with a known terminal type
        term = "xterm"

        # Get ALL capabilities from infocmp
        all_caps = self.infocmp(term)

        ncurses_code = dedent(
            f"""
            import _curses
            import json
            _curses.setupterm({repr(term)}, 1)
            results = {{}}
            for cap in {repr(all_caps)}:
                try:
                    val = _curses.tigetstr(cap)
                    if val is None:
                        results[cap] = None
                    elif val == -1:
                        results[cap] = -1
                    else:
                        results[cap] = list(val)
                except BaseException:
                    results[cap] = "error"
            print(json.dumps(results))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        self.assertEqual(
            result.returncode, 0, f"Failed to run ncurses: {result.stderr}"
        )

        ncurses_data = json.loads(result.stdout)

        ti = terminfo.TermInfo(term, fallback=False)

        # Test every single capability
        for cap in all_caps:
            if cap not in ncurses_data or ncurses_data[cap] == "error":
                continue

            with self.subTest(capability=cap):
                ncurses_val = ncurses_data[cap]
                if isinstance(ncurses_val, list):
                    ncurses_val = bytes(ncurses_val)

                pyrepl_val = ti.get(cap)

                self.assertEqual(
                    pyrepl_val,
                    ncurses_val,
                    f"Capability {cap}: ncurses={repr(ncurses_val)}, "
                    f"pyrepl={repr(pyrepl_val)}",
                )

    def test_tigetstr_input_types(self):
        """Test tigetstr with different input types."""
        term = "xterm"
        cap = "cup"

        # Test standard curses behavior with string in subprocess
        ncurses_code = dedent(
            f"""
            import _curses
            import json
            _curses.setupterm({repr(term)}, 1)

            # Test with string input
            try:
                std_str_result = _curses.tigetstr({repr(cap)})
                std_accepts_str = True
                if std_str_result is None:
                    std_str_val = None
                elif std_str_result == -1:
                    std_str_val = -1
                else:
                    std_str_val = list(std_str_result)
            except TypeError:
                std_accepts_str = False
                std_str_val = None

            print(json.dumps({{
                "accepts_str": std_accepts_str,
                "str_result": std_str_val
            }}))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        ncurses_data = json.loads(result.stdout)

        # PyREPL setup
        ti = terminfo.TermInfo(term, fallback=False)

        # PyREPL behavior with string
        try:
            pyrepl_str_result = ti.get(cap)
            pyrepl_accepts_str = True
        except TypeError:
            pyrepl_accepts_str = False

        # PyREPL should also only accept strings for compatibility
        with self.assertRaises(TypeError):
            ti.get(cap.encode("ascii"))

        # Both should accept string input
        self.assertEqual(
            pyrepl_accepts_str,
            ncurses_data["accepts_str"],
            "PyREPL and standard curses should have same string handling",
        )
        self.assertTrue(
            pyrepl_accepts_str, "PyREPL should accept string input"
        )

    def test_tparm_basic(self):
        """Test basic tparm functionality."""
        term = "xterm"
        ti = terminfo.TermInfo(term, fallback=False)

        # Test cursor positioning (cup)
        cup = ti.get("cup")
        if cup and cup not in {ABSENT_STRING, CANCELLED_STRING}:
            # Test various parameter combinations
            test_cases = [
                (0, 0),  # Top-left
                (5, 10),  # Arbitrary position
                (23, 79),  # Bottom-right of standard terminal
                (999, 999),  # Large values
            ]

            # Get ncurses results in subprocess
            ncurses_code = dedent(
                f"""
                import _curses
                import json
                _curses.setupterm({repr(term)}, 1)

                # Get cup capability
                cup = _curses.tigetstr('cup')
                results = {{}}

                for row, col in {repr(test_cases)}:
                    try:
                        result = _curses.tparm(cup, row, col)
                        results[f"{{row}},{{col}}"] = list(result)
                    except Exception as e:
                        results[f"{{row}},{{col}}"] = {{"error": str(e)}}

                print(json.dumps(results))
                """
            )

            result = subprocess.run(
                [sys.executable, "-c", ncurses_code],
                capture_output=True,
                text=True,
            )
            self.assertEqual(
                result.returncode, 0, f"Failed to run ncurses: {result.stderr}"
            )
            ncurses_data = json.loads(result.stdout)

            for row, col in test_cases:
                with self.subTest(row=row, col=col):
                    # Standard curses tparm from subprocess
                    key = f"{row},{col}"
                    if (
                        isinstance(ncurses_data[key], dict)
                        and "error" in ncurses_data[key]
                    ):
                        self.fail(
                            f"ncurses tparm failed: {ncurses_data[key]['error']}"
                        )
                    std_result = bytes(ncurses_data[key])

                    # PyREPL curses tparm
                    pyrepl_result = terminfo.tparm(cup, row, col)

                    # Results should be identical
                    self.assertEqual(
                        pyrepl_result,
                        std_result,
                        f"tparm(cup, {row}, {col}): "
                        f"std={repr(std_result)}, pyrepl={repr(pyrepl_result)}",
                    )
        else:
            raise unittest.SkipTest(
                "test_tparm_basic() requires the `cup` capability"
            )

    def test_tparm_multiple_params(self):
        """Test tparm with capabilities using multiple parameters."""
        term = "xterm"
        ti = terminfo.TermInfo(term, fallback=False)

        # Test capabilities that take parameters
        param_caps = {
            "cub": 1,  # cursor_left with count
            "cuf": 1,  # cursor_right with count
            "cuu": 1,  # cursor_up with count
            "cud": 1,  # cursor_down with count
            "dch": 1,  # delete_character with count
            "ich": 1,  # insert_character with count
        }

        # Get all capabilities from PyREPL first
        pyrepl_caps = {}
        for cap in param_caps:
            cap_value = ti.get(cap)
            if cap_value and cap_value not in {
                ABSENT_STRING,
                CANCELLED_STRING,
            }:
                pyrepl_caps[cap] = cap_value

        if not pyrepl_caps:
            self.skipTest("No parametrized capabilities found")

        # Get ncurses results in subprocess
        ncurses_code = dedent(
            f"""
            import _curses
            import json
            _curses.setupterm({repr(term)}, 1)

            param_caps = {repr(param_caps)}
            test_values = [1, 5, 10, 99]
            results = {{}}

            for cap in param_caps:
                cap_value = _curses.tigetstr(cap)
                if cap_value and cap_value != -1:
                    for value in test_values:
                        try:
                            result = _curses.tparm(cap_value, value)
                            results[f"{{cap}},{{value}}"] = list(result)
                        except Exception as e:
                            results[f"{{cap}},{{value}}"] = {{"error": str(e)}}

            print(json.dumps(results))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        self.assertEqual(
            result.returncode, 0, f"Failed to run ncurses: {result.stderr}"
        )
        ncurses_data = json.loads(result.stdout)

        for cap, cap_value in pyrepl_caps.items():
            with self.subTest(capability=cap):
                # Test with different parameter values
                for value in [1, 5, 10, 99]:
                    key = f"{cap},{value}"
                    if key in ncurses_data:
                        if (
                            isinstance(ncurses_data[key], dict)
                            and "error" in ncurses_data[key]
                        ):
                            self.fail(
                                f"ncurses tparm failed: {ncurses_data[key]['error']}"
                            )
                        std_result = bytes(ncurses_data[key])

                        pyrepl_result = terminfo.tparm(cap_value, value)
                        self.assertEqual(
                            pyrepl_result,
                            std_result,
                            f"tparm({cap}, {value}): "
                            f"std={repr(std_result)}, pyrepl={repr(pyrepl_result)}",
                        )

    def test_tparm_null_handling(self):
        """Test tparm with None/null input."""
        term = "xterm"

        ncurses_code = dedent(
            f"""
            import _curses
            import json
            _curses.setupterm({repr(term)}, 1)

            # Test with None
            try:
                _curses.tparm(None)
                raises_typeerror = False
            except TypeError:
                raises_typeerror = True
            except Exception as e:
                raises_typeerror = False
                error_type = type(e).__name__

            print(json.dumps({{"raises_typeerror": raises_typeerror}}))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        ncurses_data = json.loads(result.stdout)

        # PyREPL setup
        ti = terminfo.TermInfo(term, fallback=False)

        # Test with None - both should raise TypeError
        if ncurses_data["raises_typeerror"]:
            with self.assertRaises(TypeError):
                terminfo.tparm(None)
        else:
            # If ncurses doesn't raise TypeError, PyREPL shouldn't either
            try:
                terminfo.tparm(None)
            except TypeError:
                self.fail("PyREPL raised TypeError but ncurses did not")

    def test_special_terminals(self):
        """Test with special terminal types."""
        special_terms = [
            "dumb",  # Minimal terminal
            "unknown",  # Should fall back to defaults
            "linux",  # Linux console
            "screen",  # GNU Screen
            "tmux",  # tmux
        ]

        # Get all string capabilities from ncurses
        for term in special_terms:
            with self.subTest(term=term):
                all_caps = self.infocmp(term)
                ncurses_code = dedent(
                    f"""
                    import _curses
                    import json
                    import sys

                    try:
                        _curses.setupterm({repr(term)}, 1)
                        results = {{}}
                        for cap in {repr(all_caps)}:
                            try:
                                val = _curses.tigetstr(cap)
                                if val is None:
                                    results[cap] = None
                                elif val == -1:
                                    results[cap] = -1
                                else:
                                    # Convert bytes to list of ints for JSON
                                    results[cap] = list(val)
                            except BaseException:
                                results[cap] = "error"
                        print(json.dumps(results))
                    except Exception as e:
                        print(json.dumps({{"error": str(e)}}))
                    """
                )

                # Get ncurses results
                result = subprocess.run(
                    [sys.executable, "-c", ncurses_code],
                    capture_output=True,
                    text=True,
                )
                if result.returncode != 0:
                    self.fail(
                        f"Failed to get ncurses data for {term}: {result.stderr}"
                    )

                try:
                    ncurses_data = json.loads(result.stdout)
                except json.JSONDecodeError:
                    self.fail(
                        f"Failed to parse ncurses output for {term}: {result.stdout}"
                    )

                if "error" in ncurses_data and len(ncurses_data) == 1:
                    # ncurses failed to setup this terminal
                    # PyREPL should still work with fallback
                    ti = terminfo.TermInfo(term, fallback=True)
                    continue

                ti = terminfo.TermInfo(term, fallback=False)

                # Compare all capabilities
                for cap in all_caps:
                    if cap not in ncurses_data:
                        continue

                    with self.subTest(term=term, capability=cap):
                        ncurses_val = ncurses_data[cap]
                        if isinstance(ncurses_val, list):
                            # Convert back to bytes
                            ncurses_val = bytes(ncurses_val)

                        pyrepl_val = ti.get(cap)

                        # Both should return the same value
                        self.assertEqual(
                            pyrepl_val,
                            ncurses_val,
                            f"Capability {cap} for {term}: "
                            f"ncurses={repr(ncurses_val)}, "
                            f"pyrepl={repr(pyrepl_val)}",
                        )

    def test_terminfo_fallback(self):
        """Test that PyREPL falls back gracefully when terminfo is not found."""
        # Use a non-existent terminal type
        fake_term = "nonexistent-terminal-type-12345"

        # Check if standard curses can setup this terminal in subprocess
        ncurses_code = dedent(
            f"""
            import _curses
            import json
            try:
                _curses.setupterm({repr(fake_term)}, 1)
                print(json.dumps({{"success": True}}))
            except _curses.error:
                print(json.dumps({{"success": False, "error": "curses.error"}}))
            except Exception as e:
                print(json.dumps({{"success": False, "error": str(e)}}))
            """
        )

        result = subprocess.run(
            [sys.executable, "-c", ncurses_code],
            capture_output=True,
            text=True,
        )
        ncurses_data = json.loads(result.stdout)

        if ncurses_data["success"]:
            # If it succeeded, skip this test as we can't test fallback
            self.skipTest(
                f"System unexpectedly has terminfo for '{fake_term}'"
            )

        # PyREPL should succeed with fallback
        try:
            ti = terminfo.TermInfo(fake_term, fallback=True)
            pyrepl_ok = True
        except Exception:
            pyrepl_ok = False

        self.assertTrue(
            pyrepl_ok, "PyREPL should fall back for unknown terminals"
        )

        # Should still be able to get basic capabilities
        bel = ti.get("bel")
        self.assertIsNotNone(
            bel, "PyREPL should provide basic capabilities after fallback"
        )

    def test_invalid_terminal_names(self):
        cases = [
            (42, TypeError),
            ("", ValueError),
            ("w\x00t", ValueError),
            (f"..{os.sep}name", ValueError),
        ]

        for term, exc in cases:
            with self.subTest(term=term):
                with self.assertRaises(exc):
                    terminfo._validate_terminal_name_or_raise(term)

### File: test_unix_console.py

In [ ]:
import errno
import itertools
import os
import signal
import subprocess
import sys
import threading
import unittest
from functools import partial
from test import support
from test.support import os_helper, force_not_colorized_test_class
from test.support import script_helper, threading_helper

from unittest import TestCase
from unittest.mock import MagicMock, call, patch, ANY, Mock

from .support import handle_all_events, code_to_events

try:
    from _pyrepl.console import Event
    from _pyrepl.unix_console import UnixConsole
except ImportError:
    pass

from _pyrepl.terminfo import _TERMINAL_CAPABILITIES

TERM_CAPABILITIES = _TERMINAL_CAPABILITIES["ansi"]


def unix_console(events, **kwargs):
    console = UnixConsole(term="xterm")
    console.get_event = MagicMock(side_effect=events)
    console.getpending = MagicMock(return_value=Event("key", ""))

    height = kwargs.get("height", 25)
    width = kwargs.get("width", 80)
    console.getheightwidth = MagicMock(side_effect=lambda: (height, width))
    console.wait = MagicMock()

    console.prepare()
    for key, val in kwargs.items():
        setattr(console, key, val)
    return console


handle_events_unix_console = partial(
    handle_all_events,
    prepare_console=unix_console,
)
handle_events_narrow_unix_console = partial(
    handle_all_events,
    prepare_console=partial(unix_console, width=5),
)
handle_events_short_unix_console = partial(
    handle_all_events,
    prepare_console=partial(unix_console, height=1),
)
handle_events_unix_console_height_3 = partial(
    handle_all_events, prepare_console=partial(unix_console, height=3)
)


@unittest.skipIf(sys.platform == "win32", "No Unix event queue on Windows")
@patch(
    "_pyrepl.terminfo.tparm",
    lambda s, *args: s + b":" + b",".join(str(i).encode() for i in args),
)
@patch(
    "termios.tcgetattr",
    lambda _: [
        27394,
        3,
        19200,
        536872399,
        38400,
        38400,
        [
            b"\x04",
            b"\xff",
            b"\xff",
            b"\x7f",
            b"\x17",
            b"\x15",
            b"\x12",
            b"\x00",
            b"\x03",
            b"\x1c",
            b"\x1a",
            b"\x19",
            b"\x11",
            b"\x13",
            b"\x16",
            b"\x0f",
            b"\x01",
            b"\x00",
            b"\x14",
            b"\x00",
        ],
    ],
)
@patch("termios.tcsetattr", lambda a, b, c: None)
@patch("os.write")
@force_not_colorized_test_class
class TestConsole(TestCase):
    def test_simple_addition(self, _os_write):
        code = "12+34"
        events = code_to_events(code)
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, b"1")
        _os_write.assert_any_call(ANY, b"2")
        _os_write.assert_any_call(ANY, b"+")
        _os_write.assert_any_call(ANY, b"3")
        _os_write.assert_any_call(ANY, b"4")
        con.restore()

    def test_wrap(self, _os_write):
        code = "12+34"
        events = code_to_events(code)
        _, con = handle_events_narrow_unix_console(events)
        _os_write.assert_any_call(ANY, b"1")
        _os_write.assert_any_call(ANY, b"2")
        _os_write.assert_any_call(ANY, b"+")
        _os_write.assert_any_call(ANY, b"3")
        _os_write.assert_any_call(ANY, b"\\")
        _os_write.assert_any_call(ANY, b"\n")
        _os_write.assert_any_call(ANY, b"4")
        con.restore()

    def test_cursor_left(self, _os_write):
        code = "1"
        events = itertools.chain(
            code_to_events(code),
            [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],
        )
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cub"] + b":1")
        con.restore()

    def test_cursor_left_right(self, _os_write):
        code = "1"
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="right", raw=bytearray(b"\x1bOC")),
            ],
        )
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cub"] + b":1")
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cuf"] + b":1")
        con.restore()

    def test_cursor_up(self, _os_write):
        code = "1\n2+3"
        events = itertools.chain(
            code_to_events(code),
            [Event(evt="key", data="up", raw=bytearray(b"\x1bOA"))],
        )
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cuu"] + b":1")
        con.restore()

    def test_cursor_up_down(self, _os_write):
        code = "1\n2+3"
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cuu"] + b":1")
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cud"] + b":1")
        con.restore()

    def test_cursor_back_write(self, _os_write):
        events = itertools.chain(
            code_to_events("1"),
            [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],
            code_to_events("2"),
        )
        _, con = handle_events_unix_console(events)
        _os_write.assert_any_call(ANY, b"1")
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["cub"] + b":1")
        _os_write.assert_any_call(ANY, b"2")
        con.restore()

    def test_multiline_function_move_up_short_terminal(self, _os_write):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="scroll", data=None),
            ],
        )
        _, con = handle_events_short_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["ri"] + b":")
        con.restore()

    def test_multiline_function_move_up_down_short_terminal(self, _os_write):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="scroll", data=None),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="scroll", data=None),
            ],
        )
        _, con = handle_events_short_unix_console(events)
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["ri"] + b":")
        _os_write.assert_any_call(ANY, TERM_CAPABILITIES["ind"] + b":")
        con.restore()

    def test_resize_bigger_on_multiline_function(self, _os_write):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(code_to_events(code))
        reader, console = handle_events_short_unix_console(events)

        console.height = 2
        console.getheightwidth = MagicMock(lambda _: (2, 80))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )
        _os_write.assert_has_calls(
            [
                call(ANY, TERM_CAPABILITIES["ri"] + b":"),
                call(ANY, TERM_CAPABILITIES["cup"] + b":0,0"),
                call(ANY, b"def f():"),
            ]
        )
        console.restore()
        con.restore()

    def test_resize_smaller_on_multiline_function(self, _os_write):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(code_to_events(code))
        reader, console = handle_events_unix_console_height_3(events)

        console.height = 1
        console.getheightwidth = MagicMock(lambda _: (1, 80))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )
        _os_write.assert_has_calls(
            [
                call(ANY, TERM_CAPABILITIES["ind"] + b":"),
                call(ANY, TERM_CAPABILITIES["cup"] + b":0,0"),
                call(ANY, b"  foo"),
            ]
        )
        console.restore()
        con.restore()

    def test_getheightwidth_with_invalid_environ(self, _os_write):
        # gh-128636
        console = UnixConsole(term="xterm")
        with os_helper.EnvironmentVarGuard() as env:
            env["LINES"] = ""
            self.assertIsInstance(console.getheightwidth(), tuple)
            env["COLUMNS"] = ""
            self.assertIsInstance(console.getheightwidth(), tuple)
            os.environ = []
            self.assertIsInstance(console.getheightwidth(), tuple)

    @unittest.skipUnless(sys.platform == "darwin", "requires macOS")
    def test_restore_with_invalid_environ_on_macos(self, _os_write):
        # gh-128636 for macOS
        console = UnixConsole(term="xterm")
        with os_helper.EnvironmentVarGuard():
            os.environ = []
            console.prepare()  # needed to call restore()
            console.restore()  # this should succeed

    @threading_helper.reap_threads
    @threading_helper.requires_working_threading()
    def test_restore_in_thread(self, _os_write):
        # gh-139391: ensure that console.restore() silently suppresses
        # exceptions when calling signal.signal() from a non-main thread.
        console = unix_console([])
        console.old_sigwinch = signal.SIG_DFL
        thread = threading.Thread(target=console.restore)
        thread.start()
        thread.join()  # this should not raise


@unittest.skipIf(sys.platform == "win32", "No Unix console on Windows")
class TestUnixConsoleEIOHandling(TestCase):

    @patch('_pyrepl.unix_console.tcsetattr')
    @patch('_pyrepl.unix_console.tcgetattr')
    def test_eio_error_handling_in_restore(self, mock_tcgetattr, mock_tcsetattr):

        import termios
        mock_termios = Mock()
        mock_termios.iflag = 0
        mock_termios.oflag = 0
        mock_termios.cflag = 0
        mock_termios.lflag = 0
        mock_termios.cc = [0] * 32
        mock_termios.copy.return_value = mock_termios
        mock_tcgetattr.return_value = mock_termios

        console = UnixConsole(term="xterm")
        console.prepare()

        mock_tcsetattr.side_effect = termios.error(errno.EIO, "Input/output error")

        # EIO error should be handled gracefully in restore()
        console.restore()

    @unittest.skipUnless(sys.platform == "linux", "Only valid on Linux")
    def test_repl_eio(self):
        # Use the pty-based approach to simulate EIO error
        script_path = os.path.join(os.path.dirname(__file__), "eio_test_script.py")

        proc = script_helper.spawn_python(
            "-S", script_path,
            stderr=subprocess.PIPE,
            text=True
        )

        ready_line = proc.stdout.readline().strip()
        if ready_line != "READY" or proc.poll() is not None:
            self.fail("Child process failed to start properly")

        os.kill(proc.pid, signal.SIGUSR1)
        # sleep for pty to settle
        _, err = proc.communicate(timeout=support.LONG_TIMEOUT)
        self.assertEqual(
            proc.returncode,
            1,
            f"Expected EIO/ENXIO error, got return code {proc.returncode}",
        )
        self.assertTrue(
            (
                "Got EIO:" in err
                or "Got ENXIO:" in err
            ),
            f"Expected EIO/ENXIO error message in stderr: {err}",
        )

### File: test_utils.py

In [ ]:
from unittest import TestCase

from _pyrepl.utils import str_width, wlen, prev_next_window, gen_colors


class TestUtils(TestCase):
    def test_str_width(self):
        characters = ['a', '1', '_', '!', '\x1a', '\u263A', '\uffb9']
        for c in characters:
            self.assertEqual(str_width(c), 1)

        characters = [chr(99989), chr(99999)]
        for c in characters:
            self.assertEqual(str_width(c), 2)

    def test_wlen(self):
        for c in ['a', 'b', '1', '!', '_']:
            self.assertEqual(wlen(c), 1)
        self.assertEqual(wlen('\x1a'), 2)

        char_east_asian_width_N = chr(3800)
        self.assertEqual(wlen(char_east_asian_width_N), 1)
        char_east_asian_width_W = chr(4352)
        self.assertEqual(wlen(char_east_asian_width_W), 2)

        self.assertEqual(wlen('hello'), 5)
        self.assertEqual(wlen('hello' + '\x1a'), 7)

    def test_prev_next_window(self):
        def gen_normal():
            yield 1
            yield 2
            yield 3
            yield 4

        pnw = prev_next_window(gen_normal())
        self.assertEqual(next(pnw), (None, 1, 2))
        self.assertEqual(next(pnw), (1, 2, 3))
        self.assertEqual(next(pnw), (2, 3, 4))
        self.assertEqual(next(pnw), (3, 4, None))
        with self.assertRaises(StopIteration):
            next(pnw)

        def gen_short():
            yield 1

        pnw = prev_next_window(gen_short())
        self.assertEqual(next(pnw), (None, 1, None))
        with self.assertRaises(StopIteration):
            next(pnw)

        def gen_raise():
            yield from gen_normal()
            1/0

        pnw = prev_next_window(gen_raise())
        self.assertEqual(next(pnw), (None, 1, 2))
        self.assertEqual(next(pnw), (1, 2, 3))
        self.assertEqual(next(pnw), (2, 3, 4))
        self.assertEqual(next(pnw), (3, 4, None))
        with self.assertRaises(ZeroDivisionError):
            next(pnw)

    def test_gen_colors_keyword_highlighting(self):
        cases = [
            # no highlights
            ("a.set", [(".", "op")]),
            ("obj.list", [(".", "op")]),
            ("obj.match", [(".", "op")]),
            ("b. \\\n format", [(".", "op")]),
            # highlights
            ("set", [("set", "builtin")]),
            ("list", [("list", "builtin")]),
            ("    \n dict", [("dict", "builtin")]),
        ]
        for code, expected_highlights in cases:
            with self.subTest(code=code):
                colors = list(gen_colors(code))
                # Extract (text, tag) pairs for comparison
                actual_highlights = []
                for color in colors:
                    span_text = code[color.span.start:color.span.end + 1]
                    actual_highlights.append((span_text, color.tag))
                self.assertEqual(actual_highlights, expected_highlights)

### File: test_windows_console.py

In [ ]:
import sys
import unittest

if sys.platform != "win32":
    raise unittest.SkipTest("test only relevant on win32")


import itertools
from functools import partial
from test.support import force_not_colorized_test_class
from typing import Iterable
from unittest import TestCase
from unittest.mock import MagicMock, call

from .support import handle_all_events, code_to_events
from .support import prepare_reader as default_prepare_reader

try:
    from _pyrepl.console import Event, Console
    from _pyrepl.windows_console import (
        WindowsConsole,
        MOVE_LEFT,
        MOVE_RIGHT,
        MOVE_UP,
        MOVE_DOWN,
        ERASE_IN_LINE,
    )
    import _pyrepl.windows_console as wc
except ImportError:
    pass


@force_not_colorized_test_class
class WindowsConsoleTests(TestCase):
    def console(self, events, **kwargs) -> Console:
        console = WindowsConsole()
        console.get_event = MagicMock(side_effect=events)
        console.getpending = MagicMock(return_value=Event("key", ""))
        console.wait = MagicMock()
        console._scroll = MagicMock()
        console._hide_cursor = MagicMock()
        console._show_cursor = MagicMock()
        console._getscrollbacksize = MagicMock(42)
        console.out = MagicMock()

        height = kwargs.get("height", 25)
        width = kwargs.get("width", 80)
        console.getheightwidth = MagicMock(side_effect=lambda: (height, width))

        console.prepare()
        for key, val in kwargs.items():
            setattr(console, key, val)
        return console

    def handle_events(
        self,
        events: Iterable[Event],
        prepare_console=None,
        prepare_reader=None,
        **kwargs,
    ):
        prepare_console = prepare_console or partial(self.console, **kwargs)
        prepare_reader = prepare_reader or default_prepare_reader
        return handle_all_events(events, prepare_console, prepare_reader)

    def handle_events_narrow(self, events):
        return self.handle_events(events, width=5)

    def handle_events_short(self, events, **kwargs):
        return self.handle_events(events, height=1, **kwargs)

    def handle_events_height_3(self, events):
        return self.handle_events(events, height=3)

    def test_simple_addition(self):
        code = "12+34"
        events = code_to_events(code)
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(b"1")
        con.out.write.assert_any_call(b"2")
        con.out.write.assert_any_call(b"+")
        con.out.write.assert_any_call(b"3")
        con.out.write.assert_any_call(b"4")
        con.restore()

    def test_wrap(self):
        code = "12+34"
        events = code_to_events(code)
        _, con = self.handle_events_narrow(events)
        con.out.write.assert_any_call(b"1")
        con.out.write.assert_any_call(b"2")
        con.out.write.assert_any_call(b"+")
        con.out.write.assert_any_call(b"3")
        con.out.write.assert_any_call(b"\\")
        con.out.write.assert_any_call(b"\n")
        con.out.write.assert_any_call(b"4")
        con.restore()

    def test_resize_wider(self):
        code = "1234567890"
        events = code_to_events(code)
        reader, console = self.handle_events_narrow(events)

        console.height = 20
        console.width = 80
        console.getheightwidth = MagicMock(lambda _: (20, 80))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )

        con.out.write.assert_any_call(self.move_right(2))
        con.out.write.assert_any_call(self.move_up(2))
        con.out.write.assert_any_call(b"567890")

        con.restore()

    def test_resize_narrower(self):
        code = "1234567890"
        events = code_to_events(code)
        reader, console = self.handle_events(events)

        console.height = 20
        console.width = 4
        console.getheightwidth = MagicMock(lambda _: (20, 4))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )

        con.out.write.assert_any_call(b"456\\")
        con.out.write.assert_any_call(b"789\\")

        con.restore()

    def test_cursor_left(self):
        code = "1"
        events = itertools.chain(
            code_to_events(code),
            [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],
        )
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(self.move_left())
        con.restore()

    def test_cursor_left_right(self):
        code = "1"
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="left", raw=bytearray(b"\x1bOD")),
                Event(evt="key", data="right", raw=bytearray(b"\x1bOC")),
            ],
        )
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(self.move_left())
        con.out.write.assert_any_call(self.move_right())
        con.restore()

    def test_cursor_up(self):
        code = "1\n2+3"
        events = itertools.chain(
            code_to_events(code),
            [Event(evt="key", data="up", raw=bytearray(b"\x1bOA"))],
        )
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(self.move_up())
        con.restore()

    def test_cursor_up_down(self):
        code = "1\n2+3"
        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
            ],
        )
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(self.move_up())
        con.out.write.assert_any_call(self.move_down())
        con.restore()

    def test_cursor_back_write(self):
        events = itertools.chain(
            code_to_events("1"),
            [Event(evt="key", data="left", raw=bytearray(b"\x1bOD"))],
            code_to_events("2"),
        )
        _, con = self.handle_events(events)
        con.out.write.assert_any_call(b"1")
        con.out.write.assert_any_call(self.move_left())
        con.out.write.assert_any_call(b"21")
        con.restore()

    def test_multiline_function_move_up_short_terminal(self):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="scroll", data=None),
            ],
        )
        _, con = self.handle_events_short(events)
        con.out.write.assert_any_call(self.move_left(5))
        con.out.write.assert_any_call(self.move_up())
        con.restore()

    def test_multiline_function_move_up_down_short_terminal(self):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data="up", raw=bytearray(b"\x1bOA")),
                Event(evt="scroll", data=None),
                Event(evt="key", data="down", raw=bytearray(b"\x1bOB")),
                Event(evt="scroll", data=None),
            ],
        )
        _, con = self.handle_events_short(events)
        con.out.write.assert_any_call(self.move_left(8))
        con.out.write.assert_any_call(self.erase_in_line())
        con.restore()

    def test_resize_bigger_on_multiline_function(self):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(code_to_events(code))
        reader, console = self.handle_events_short(events)

        console.height = 2
        console.getheightwidth = MagicMock(lambda _: (2, 80))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )
        con.out.write.assert_has_calls(
            [
                call(self.move_left(5)),
                call(self.move_up()),
                call(b"def f():"),
                call(self.move_left(3)),
                call(self.move_down()),
            ]
        )
        console.restore()
        con.restore()

    def test_resize_smaller_on_multiline_function(self):
        # fmt: off
        code = (
            "def f():\n"
            "  foo"
        )
        # fmt: on

        events = itertools.chain(code_to_events(code))
        reader, console = self.handle_events_height_3(events)

        console.height = 1
        console.getheightwidth = MagicMock(lambda _: (1, 80))

        def same_reader(_):
            return reader

        def same_console(events):
            console.get_event = MagicMock(side_effect=events)
            return console

        _, con = handle_all_events(
            [Event(evt="resize", data=None)],
            prepare_reader=same_reader,
            prepare_console=same_console,
        )
        con.out.write.assert_has_calls(
            [
                call(self.move_left(5)),
                call(self.move_up()),
                call(self.erase_in_line()),
                call(b"  foo"),
            ]
        )
        console.restore()
        con.restore()

    def move_up(self, lines=1):
        return MOVE_UP.format(lines).encode("utf8")

    def move_down(self, lines=1):
        return MOVE_DOWN.format(lines).encode("utf8")

    def move_left(self, cols=1):
        return MOVE_LEFT.format(cols).encode("utf8")

    def move_right(self, cols=1):
        return MOVE_RIGHT.format(cols).encode("utf8")

    def erase_in_line(self):
        return ERASE_IN_LINE.encode("utf8")

    def test_multiline_ctrl_z(self):
        # see gh-126332
        code = "abcdefghi"

        events = itertools.chain(
            code_to_events(code),
            [
                Event(evt="key", data='\x1a', raw=bytearray(b'\x1a')),
                Event(evt="key", data='\x1a', raw=bytearray(b'\x1a')),
            ],
        )
        reader, con = self.handle_events_narrow(events)
        self.assertEqual(reader.cxy, (2, 3))
        con.restore()


class WindowsConsoleGetEventTests(TestCase):
    # Virtual-Key Codes: https://learn.microsoft.com/en-us/windows/win32/inputdev/virtual-key-codes
    VK_BACK = 0x08
    VK_RETURN = 0x0D
    VK_LEFT = 0x25
    VK_7 = 0x37
    VK_M = 0x4D
    # Used for miscellaneous characters; it can vary by keyboard.
    # For the US standard keyboard, the '" key.
    # For the German keyboard, the Ä key.
    VK_OEM_7 = 0xDE

    # State of control keys: https://learn.microsoft.com/en-us/windows/console/key-event-record-str
    RIGHT_ALT_PRESSED = 0x0001
    RIGHT_CTRL_PRESSED = 0x0004
    LEFT_ALT_PRESSED = 0x0002
    LEFT_CTRL_PRESSED = 0x0008
    ENHANCED_KEY = 0x0100
    SHIFT_PRESSED = 0x0010


    def get_event(self, input_records, **kwargs) -> Console:
        self.console = WindowsConsole(encoding='utf-8')
        self.mock = MagicMock(side_effect=input_records)
        self.console._read_input = self.mock
        self.console._WindowsConsole__vt_support = kwargs.get("vt_support",
                                                              False)
        self.console.wait = MagicMock(return_value=True)
        event = self.console.get_event(block=False)
        return event

    def get_input_record(self, unicode_char, vcode=0, control=0):
        return wc.INPUT_RECORD(
            wc.KEY_EVENT,
            wc.ConsoleEvent(KeyEvent=
                wc.KeyEvent(
                    bKeyDown=True,
                    wRepeatCount=1,
                    wVirtualKeyCode=vcode,
                    wVirtualScanCode=0, # not used
                    uChar=wc.Char(unicode_char),
                    dwControlKeyState=control
                    )))

    def test_EmptyBuffer(self):
        self.assertEqual(self.get_event([None]), None)
        self.assertEqual(self.mock.call_count, 1)

    def test_WINDOW_BUFFER_SIZE_EVENT(self):
        ir = wc.INPUT_RECORD(
            wc.WINDOW_BUFFER_SIZE_EVENT,
            wc.ConsoleEvent(WindowsBufferSizeEvent=
                wc.WindowsBufferSizeEvent(
                    wc._COORD(0, 0))))
        self.assertEqual(self.get_event([ir]), Event("resize", ""))
        self.assertEqual(self.mock.call_count, 1)

    def test_KEY_EVENT_up_ignored(self):
        ir = wc.INPUT_RECORD(
            wc.KEY_EVENT,
            wc.ConsoleEvent(KeyEvent=
                wc.KeyEvent(bKeyDown=False)))
        self.assertEqual(self.get_event([ir]), None)
        self.assertEqual(self.mock.call_count, 1)

    def test_unhandled_events(self):
        for event in (wc.FOCUS_EVENT, wc.MENU_EVENT, wc.MOUSE_EVENT):
            ir = wc.INPUT_RECORD(
                event,
                # fake data, nothing is read except bKeyDown
                wc.ConsoleEvent(KeyEvent=
                    wc.KeyEvent(bKeyDown=False)))
            self.assertEqual(self.get_event([ir]), None)
            self.assertEqual(self.mock.call_count, 1)

    def test_enter(self):
        ir = self.get_input_record("\r", self.VK_RETURN)
        self.assertEqual(self.get_event([ir]), Event("key", "\n"))
        self.assertEqual(self.mock.call_count, 1)

    def test_backspace(self):
        ir = self.get_input_record("\x08", self.VK_BACK)
        self.assertEqual(
            self.get_event([ir]), Event("key", "backspace"))
        self.assertEqual(self.mock.call_count, 1)

    def test_m(self):
        ir = self.get_input_record("m", self.VK_M)
        self.assertEqual(self.get_event([ir]), Event("key", "m"))
        self.assertEqual(self.mock.call_count, 1)

    def test_M(self):
        ir = self.get_input_record("M", self.VK_M, self.SHIFT_PRESSED)
        self.assertEqual(self.get_event([ir]), Event("key", "M"))
        self.assertEqual(self.mock.call_count, 1)

    def test_left(self):
        # VK_LEFT is sent as ENHANCED_KEY
        ir = self.get_input_record("\x00", self.VK_LEFT, self.ENHANCED_KEY)
        self.assertEqual(self.get_event([ir]), Event("key", "left"))
        self.assertEqual(self.mock.call_count, 1)

    def test_left_RIGHT_CTRL_PRESSED(self):
        ir = self.get_input_record(
            "\x00", self.VK_LEFT, self.RIGHT_CTRL_PRESSED | self.ENHANCED_KEY)
        self.assertEqual(
            self.get_event([ir]), Event("key", "ctrl left"))
        self.assertEqual(self.mock.call_count, 1)

    def test_left_LEFT_CTRL_PRESSED(self):
        ir = self.get_input_record(
            "\x00", self.VK_LEFT, self.LEFT_CTRL_PRESSED | self.ENHANCED_KEY)
        self.assertEqual(
            self.get_event([ir]), Event("key", "ctrl left"))
        self.assertEqual(self.mock.call_count, 1)

    def test_left_RIGHT_ALT_PRESSED(self):
        ir = self.get_input_record(
            "\x00", self.VK_LEFT, self.RIGHT_ALT_PRESSED | self.ENHANCED_KEY)
        self.assertEqual(self.get_event([ir]), Event(evt="key", data="\033"))
        self.assertEqual(
            self.console.get_event(), Event("key", "left"))
        # self.mock is not called again, since the second time we read from the
        # command queue
        self.assertEqual(self.mock.call_count, 1)

    def test_left_LEFT_ALT_PRESSED(self):
        ir = self.get_input_record(
            "\x00", self.VK_LEFT, self.LEFT_ALT_PRESSED | self.ENHANCED_KEY)
        self.assertEqual(self.get_event([ir]), Event(evt="key", data="\033"))
        self.assertEqual(
            self.console.get_event(), Event("key", "left"))
        self.assertEqual(self.mock.call_count, 1)

    def test_m_LEFT_ALT_PRESSED_and_LEFT_CTRL_PRESSED(self):
        # For the shift keys, Windows does not send anything when
        # ALT and CTRL are both pressed, so let's test with VK_M.
        # get_event() receives this input, but does not
        # generate an event.
        # This is for e.g. an English keyboard layout, for a
        # German layout this returns `µ`, see test_AltGr_m.
        ir = self.get_input_record(
            "\x00", self.VK_M, self.LEFT_ALT_PRESSED | self.LEFT_CTRL_PRESSED)
        self.assertEqual(self.get_event([ir]), None)
        self.assertEqual(self.mock.call_count, 1)

    def test_m_LEFT_ALT_PRESSED(self):
        ir = self.get_input_record(
            "m", vcode=self.VK_M, control=self.LEFT_ALT_PRESSED)
        self.assertEqual(self.get_event([ir]), Event(evt="key", data="\033"))
        self.assertEqual(self.console.get_event(), Event("key", "m"))
        self.assertEqual(self.mock.call_count, 1)

    def test_m_RIGHT_ALT_PRESSED(self):
        ir = self.get_input_record(
            "m", vcode=self.VK_M, control=self.RIGHT_ALT_PRESSED)
        self.assertEqual(self.get_event([ir]), Event(evt="key", data="\033"))
        self.assertEqual(self.console.get_event(), Event("key", "m"))
        self.assertEqual(self.mock.call_count, 1)

    def test_AltGr_7(self):
        # E.g. on a German keyboard layout, '{' is entered via
        # AltGr + 7, where AltGr is the right Alt key on the keyboard.
        # In this case, Windows automatically sets
        # RIGHT_ALT_PRESSED = 0x0001 + LEFT_CTRL_PRESSED = 0x0008
        # This can also be entered like
        # LeftAlt + LeftCtrl + 7 or
        # LeftAlt + RightCtrl + 7
        # See https://learn.microsoft.com/en-us/windows/console/key-event-record-str
        # https://learn.microsoft.com/en-us/windows/win32/api/winuser/nf-winuser-vkkeyscanw
        ir = self.get_input_record(
            "{", vcode=self.VK_7,
            control=self.RIGHT_ALT_PRESSED | self.LEFT_CTRL_PRESSED)
        self.assertEqual(self.get_event([ir]), Event("key", "{"))
        self.assertEqual(self.mock.call_count, 1)

    def test_AltGr_m(self):
        # E.g. on a German keyboard layout, this yields 'µ'
        # Let's use LEFT_ALT_PRESSED and RIGHT_CTRL_PRESSED this
        # time, to cover that, too. See above in test_AltGr_7.
        ir = self.get_input_record(
            "µ", vcode=self.VK_M, control=self.LEFT_ALT_PRESSED | self.RIGHT_CTRL_PRESSED)
        self.assertEqual(self.get_event([ir]), Event("key", "µ"))
        self.assertEqual(self.mock.call_count, 1)

    def test_umlaut_a_german(self):
        ir = self.get_input_record("ä", self.VK_OEM_7)
        self.assertEqual(self.get_event([ir]), Event("key", "ä"))
        self.assertEqual(self.mock.call_count, 1)

    # virtual terminal tests
    # Note: wVirtualKeyCode, wVirtualScanCode and dwControlKeyState
    # are always zero in this case.
    # "\r" and backspace are handled specially, everything else
    # is handled in "elif self.__vt_support:" in WindowsConsole.get_event().
    # Hence, only one regular key ("m") and a terminal sequence
    # are sufficient to test here, the real tests happen in test_eventqueue
    # and test_keymap.

    def test_enter_vt(self):
        ir = self.get_input_record("\r")
        self.assertEqual(self.get_event([ir], vt_support=True),
                         Event("key", "\n"))
        self.assertEqual(self.mock.call_count, 1)

    def test_backspace_vt(self):
        ir = self.get_input_record("\x7f")
        self.assertEqual(self.get_event([ir], vt_support=True),
                         Event("key", "backspace", b"\x7f"))
        self.assertEqual(self.mock.call_count, 1)

    def test_up_vt(self):
        irs = [self.get_input_record(x) for x in "\x1b[A"]
        self.assertEqual(self.get_event(irs, vt_support=True),
                         Event(evt='key', data='up', raw=bytearray(b'\x1b[A')))
        self.assertEqual(self.mock.call_count, 3)


if __name__ == "__main__":
    unittest.main()